In [ ]:
# =============================================================
# CELL 0: Setup + load reference data
# =============================================================
import io
import json
import os
import zipfile
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd
import revelio
import revelio.base

sfClient = revelio.base.client('snowflake', 'caelan@reveliolabs.com')


def normalize_parquet_dtypes(df):
    """Convert Snowflake Decimal object columns before PyArrow parquet export."""
    out = df.copy()
    for col in out.columns:
        if out[col].dtype != 'object':
            continue
        non_null = out[col].dropna()
        if non_null.empty:
            continue
        if non_null.map(lambda v: isinstance(v, Decimal)).any():
            out[col] = out[col].map(lambda v: float(v) if isinstance(v, Decimal) else v)
            non_null = out[col].dropna()
        if not non_null.empty and non_null.map(lambda v: isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)).all():
            out[col] = pd.to_numeric(out[col], errors='coerce')
    return out


def write_fact_parquet(df, path):
    normalize_parquet_dtypes(df).to_parquet(path, index=False)

SCRATCH = 'USER_CAELAN.TMP_MONTHLY'
EDUCATION_CIP = os.environ.get('OUTCOMES_EDUCATION_CIP', f'{SCRATCH}.EDUCATION_WITH_CIP')
POSITION_TABLE = os.environ.get('OUTCOMES_POSITION_TABLE', 'CLIENT_STANDARD.REVELIO_INTERNAL.STANDARD_202603_INDIVIDUAL_POSITION')
USER_TABLE = os.environ.get('OUTCOMES_USER_TABLE', 'CLIENT_STANDARD.REVELIO_INTERNAL.STANDARD_202603_INDIVIDUAL_USER')
CURRENT_RAW_EDUCATION_TABLE = os.environ.get('OUTCOMES_CURRENT_RAW_EDUCATION_TABLE', 'SERVICE_PIPELINES.OUTPUT_CURRENT.INDIVIDUAL_EDUCATION_UNIFIED')
print(f'Using education-with-CIP source: {EDUCATION_CIP}')
print(f'Using position source: {POSITION_TABLE}')
print(f'Current raw education reference: {CURRENT_RAW_EDUCATION_TABLE}')


position_schema_probe = sfClient.load_df(f"SELECT * FROM {POSITION_TABLE} LIMIT 0")
POSITION_COLUMNS = {c.lower(): c for c in position_schema_probe.columns}
POSITION_WEIGHT_CANDIDATES = [
    'position_weight', 'position_weights', 'sample_weight', 'sample_weights',
    'universe_weight', 'representation_weight', 'profile_weight', 'individual_weight',
    'person_weight', 'user_weight', 'weight'
]
POSITION_WEIGHT_COL = next((POSITION_COLUMNS[c] for c in POSITION_WEIGHT_CANDIDATES if c in POSITION_COLUMNS), None)
print(f'Position table columns: {len(POSITION_COLUMNS)}')
print(f'Position weight column: {POSITION_WEIGHT_COL or "none found; using 1.0"}')

def position_weight_sql(alias='p'):
    if POSITION_WEIGHT_COL:
        return f"COALESCE(TRY_TO_DOUBLE({alias}.{POSITION_WEIGHT_COL}), 1.0)"
    return "1.0"

HORIZONS = [1, 5, 10]


def _safe_precompute_work_dir():
    configured = os.environ.get('OUTCOMES_PRECOMPUTE_WORK_DIR')
    if configured:
        return Path(configured).expanduser().resolve()
    fallback_candidates = [
        Path('~/data0_caelan/nace_june5_503_precompute_server').expanduser(),
        Path('/data0/data0_caelan/nace_june5_503_precompute_server'),
        Path.cwd() if Path.cwd().exists() else None,
        Path.home(),
    ]
    for candidate in fallback_candidates:
        if candidate is None:
            continue
        try:
            resolved = candidate.resolve()
            resolved.mkdir(parents=True, exist_ok=True)
            return resolved
        except (FileNotFoundError, PermissionError, OSError):
            continue
    return Path.home().resolve()


WORK_DIR = _safe_precompute_work_dir()
WORK_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path(os.environ.get('OUTCOMES_PRECOMPUTE_OUT_DIR', 'school_outcomes_data_nace_june5_503_plus_elite')).expanduser()
if not OUT_DIR.is_absolute():
    OUT_DIR = WORK_DIR / OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path(os.environ.get('OUTCOMES_PRECOMPUTE_DATA_DIR', 'csvs')).expanduser()
if not DATA_DIR.is_absolute():
    DATA_DIR = WORK_DIR / DATA_DIR
DATA_DIR = DATA_DIR.resolve()
OUTCOME_DEGREES = ['Associates', 'Bachelors', 'Masters', 'Research Doctorate', 'Professional Doctorate', 'Other Doctorate']
CIP_CORRECTION_PATHS = [
    DATA_DIR / 'cip_match_corrections.csv',
    Path('cip_match_corrections.csv'),
]
# Current v4.3 EDUCATION_WITH_CIP stores the final assigned major at CIP4.
# Older outcomes notebooks treated cip_code as CIP6, so keep CIP4 primary
# and expose CIP6 only as a top-candidate diagnostic when present.
education_schema_probe = sfClient.load_df(f"SELECT * FROM {EDUCATION_CIP} LIMIT 0")
EDUCATION_COLUMNS = {c.lower() for c in education_schema_probe.columns}
print(f'Education CIP table columns: {len(EDUCATION_COLUMNS)}')

def _qualified_col(alias, col):
    return f"{alias}.{col}" if alias else col

def _available_expr(alias, candidates):
    exprs = [f"NULLIF(TO_VARCHAR({_qualified_col(alias, col)}), '')" for col in candidates if col.lower() in EDUCATION_COLUMNS]
    if not exprs:
        return "CAST(NULL AS VARCHAR)"
    if len(exprs) == 1:
        return exprs[0]
    return f"COALESCE({', '.join(exprs)})"

def assigned_cip4_sql(alias=''):
    raw = _available_expr(alias, ['outcomes_cip4_code', 'cip_code', 'top_cip4_code', 'alloc_cip4_code'])
    return f"CASE WHEN LENGTH({raw}) = 7 AND SUBSTR({raw}, 3, 1) = '.' THEN SUBSTR({raw}, 1, 5) ELSE {raw} END"

def candidate_cip6_sql(alias=''):
    if 'top_cip6_code' in EDUCATION_COLUMNS:
        return f"NULLIF(TO_VARCHAR({_qualified_col(alias, 'top_cip6_code')}), '')"
    raw = _available_expr(alias, ['cip_code'])
    return f"CASE WHEN LENGTH({raw}) = 7 AND SUBSTR({raw}, 3, 1) = '.' THEN {raw} ELSE CAST(NULL AS VARCHAR) END"

def degree_cip_sql(alias=''):
    return f"COALESCE({candidate_cip6_sql(alias)}, {assigned_cip4_sql(alias)})"

def assigned_cip_title_sql(alias=''):
    if 'cip_title' in EDUCATION_COLUMNS:
        return f"COALESCE(NULLIF(TO_VARCHAR({_qualified_col(alias, 'cip_title')}), ''), '')"
    return "''"

required_education_cols = {'user_id', 'unitid', 'ipeds_name', 'degree', 'enddate'}
missing_required_education_cols = sorted(required_education_cols - EDUCATION_COLUMNS)
if missing_required_education_cols:
    raise ValueError(f'Missing required EDUCATION_WITH_CIP columns: {missing_required_education_cols}')
if not ({'cip_code', 'top_cip4_code', 'alloc_cip4_code'} & EDUCATION_COLUMNS):
    raise ValueError('No usable CIP4 column found; expected cip_code, top_cip4_code, or alloc_cip4_code')

def _format_cip4_code(value):
    if pd.isna(value):
        return ''
    text = str(value).strip().replace('"', '')
    if not text or text.lower() == 'nan':
        return ''
    try:
        text = f'{float(text):07.4f}'
    except ValueError:
        pass
    if len(text) == 7 and text[2] == '.':
        return text[:5]
    if len(text) == 5 and text[2] == '.':
        return text
    return ''

def _normalize_major_key(series):
    return (
        series.fillna('')
        .astype(str)
        .str.lower()
        .str.replace(r'[^a-z0-9]+', ' ', regex=True)
        .str.strip()
    )

def load_cip_corrections():
    for path in CIP_CORRECTION_PATHS:
        if Path(path).exists():
            raw = pd.read_csv(path, dtype=str)
            required_cols = {'major_norm', 'action', 'cip_code'}
            if not required_cols.issubset(raw.columns):
                print(f'CIP corrections skipped; {path} missing columns {sorted(required_cols - set(raw.columns))}')
                return pd.DataFrame(columns=['major_norm', 'corrected_cip4'])
            fixes = raw[raw['action'].fillna('').str.lower().eq('fix')].copy()
            fixes['major_norm'] = _normalize_major_key(fixes['major_norm'])
            fixes['corrected_cip4'] = fixes['cip_code'].apply(_format_cip4_code)
            fixes = fixes[(fixes['major_norm'] != '') & (fixes['corrected_cip4'] != '')]
            fixes = fixes[['major_norm', 'corrected_cip4']].drop_duplicates('major_norm', keep='last')
            print(f'Loaded {len(fixes):,} CIP correction rows from {path}')
            return fixes.reset_index(drop=True)
    print('No cip_match_corrections.csv found; using assigned CIP fields as-is.')
    return pd.DataFrame(columns=['major_norm', 'corrected_cip4'])

cip_corrections = load_cip_corrections()
if len(cip_corrections) and 'major_norm' in EDUCATION_COLUMNS:
    correction_table = f'{SCRATCH}.CIP_MATCH_CORRECTIONS'
    corrected_education_table = f'{SCRATCH}.EDUCATION_WITH_CIP_OUTCOMES'
    conn = sfClient.connect()
    cur = conn.cursor()
    cur.execute(f"CREATE OR REPLACE TABLE {correction_table} (major_norm VARCHAR, corrected_cip4 VARCHAR)")
    cur.executemany(
        f"INSERT INTO {correction_table} (major_norm, corrected_cip4) VALUES (%s, %s)",
        list(cip_corrections.itertuples(index=False, name=None)),
    )
    source_education_cip = EDUCATION_CIP
    cur.execute(f"""
        CREATE OR REPLACE TABLE {corrected_education_table} AS
        SELECT
            e.*,
            COALESCE(c.corrected_cip4, {assigned_cip4_sql('e')}) AS outcomes_cip4_code,
            c.corrected_cip4 AS manual_corrected_cip4,
            CASE WHEN c.corrected_cip4 IS NOT NULL THEN 1 ELSE 0 END AS cip_manual_override_flag
        FROM {source_education_cip} e
        LEFT JOIN {correction_table} c
          ON LOWER(TRIM(TO_VARCHAR(e.major_norm))) = c.major_norm
    """)
    EDUCATION_CIP = corrected_education_table
    education_schema_probe = sfClient.load_df(f"SELECT * FROM {EDUCATION_CIP} LIMIT 0")
    EDUCATION_COLUMNS = {c.lower() for c in education_schema_probe.columns}
    print(f'Applied CIP corrections through {corrected_education_table}.')
elif len(cip_corrections):
    print('CIP corrections available, but EDUCATION_WITH_CIP has no major_norm column; skipping overrides.')
print('Primary major key for outcomes: assigned CIP4; top_cip6_code is diagnostic when available.')

# --- Apply Revelio school-entity -> IPEDS UnitID repairs before any school filters ---
# Some prominent campuses (UCs, UT Austin, UMBC) exist in the Revelio school
# entity mapping but can be missing or zeroed in EDUCATION_WITH_CIP.unitid. If
# the education table exposes entity IDs or raw school names, repair unitid here
# so all downstream cells count the school instead of silently dropping rows.
INSTITUTION_MAPPING_PATHS = [
    DATA_DIR / 'revelio_ipeds_institution_mapping.csv',
    DATA_DIR / 'revelio_ipeds_school_name_map_labeled_20260520.csv',
    DATA_DIR / 'education_school_entity_to_unitid_lookup_v4_3_0.csv',
    DATA_DIR / 'school_entity_crosswalk_corrected_full_v4_3_0.csv',
    Path('revelio_ipeds_institution_mapping.csv'),
    Path('revelio_ipeds_school_name_map_labeled_20260520.csv'),
    Path('education_school_entity_to_unitid_lookup_v4_3_0.csv'),
    Path('school_entity_crosswalk_corrected_full_v4_3_0.csv'),
]

PROMINENT_SCHOOL_NAME_UNITID_OVERRIDES = [
    ('University of California Berkeley', '110635', 'University of California-Berkeley'),
    ('University of California, Berkeley', '110635', 'University of California-Berkeley'),
    ('UC Berkeley', '110635', 'University of California-Berkeley'),
    ('University of California Davis', '110644', 'University of California-Davis'),
    ('UC Davis', '110644', 'University of California-Davis'),
    ('University of California Irvine', '110653', 'University of California-Irvine'),
    ('UC Irvine', '110653', 'University of California-Irvine'),
    ('University of California Los Angeles', '110662', 'University of California-Los Angeles'),
    ('University of California, Los Angeles', '110662', 'University of California-Los Angeles'),
    ('University of California-Los Angeles', '110662', 'University of California-Los Angeles'),
    ('UCLA', '110662', 'University of California-Los Angeles'),
    ('University of California Merced', '445188', 'University of California-Merced'),
    ('UC Merced', '445188', 'University of California-Merced'),
    ('University of California Riverside', '110671', 'University of California-Riverside'),
    ('UC Riverside', '110671', 'University of California-Riverside'),
    ('University of California San Diego', '110680', 'University of California-San Diego'),
    ('UC San Diego', '110680', 'University of California-San Diego'),
    ('UCSD', '110680', 'University of California-San Diego'),
    ('University of California Santa Barbara', '110705', 'University of California-Santa Barbara'),
    ('UC Santa Barbara', '110705', 'University of California-Santa Barbara'),
    ('UCSB', '110705', 'University of California-Santa Barbara'),
    ('University of California Santa Cruz', '110714', 'University of California-Santa Cruz'),
    ('UC Santa Cruz', '110714', 'University of California-Santa Cruz'),
    ('UCSC', '110714', 'University of California-Santa Cruz'),
    ('University of Maryland Baltimore County', '163268', 'University of Maryland-Baltimore County'),
    ('University of Maryland-Baltimore County', '163268', 'University of Maryland-Baltimore County'),
    ('UMBC', '163268', 'University of Maryland-Baltimore County'),
    ('The University of Texas at Austin', '228778', 'The University of Texas at Austin'),
    ('University of Texas at Austin', '228778', 'The University of Texas at Austin'),
    ('UT Austin', '228778', 'The University of Texas at Austin'),
    ('UC-Berkeley', '110635', 'University of California-Berkeley'),
    ('UT-Austin', '228778', 'The University of Texas at Austin'),
    ('Amarillo College', '222576', 'Amarillo College'),
    ('Connecticut State Community College', '129367', 'Connecticut State Community College'),
    ('CT State Community College', '129367', 'Connecticut State Community College'),
    ('CT State Community College Housatonic', '129367', 'Connecticut State Community College'),
    ('CT State Community College - Housatonic', '129367', 'Connecticut State Community College'),
    ('Connecticut State Community College Housatonic', '129367', 'Connecticut State Community College'),
    ('Housatonic Community College', '129367', 'Connecticut State Community College'),
    ('Washington University in St Louis', '179867', 'Washington University in St Louis'),
    ('Washington University in St. Louis', '179867', 'Washington University in St Louis'),
    ('WashU', '179867', 'Washington University in St Louis'),
    ('California State University Sacramento', '110617', 'California State University-Sacramento'),
    ('California State University - Sacramento', '110617', 'California State University-Sacramento'),
    ('California State University-Sacramento', '110617', 'California State University-Sacramento'),
    ('CSU Sacramento', '110617', 'California State University-Sacramento'),
    ('Sacramento State', '110617', 'California State University-Sacramento'),
    ('Columbia Basin College', '234979', 'Columbia Basin College'),
    ('Madison Area Technical College', '238263', 'Madison Area Technical College'),
    ('Madison College', '238263', 'Madison Area Technical College'),
    ('Pellissippi State Community College', '221643', 'Pellissippi State Community College'),
    ('Arizona State University', '104151', 'Arizona State University Campus Immersion'),
    ('ASU', '104151', 'Arizona State University Campus Immersion'),
    # Penn State branch campuses. Keep broad "Penn State University" mapped to Main
    # Campus; only these branch-specific raw names should split out.
    ('Pennsylvania State University Penn State Erie Behrend College', '214591', 'Pennsylvania State University-Penn State Erie-Behrend College'),
    ('Pennsylvania State University-Penn State Erie-Behrend College', '214591', 'Pennsylvania State University-Penn State Erie-Behrend College'),
    ('Penn State Erie The Behrend College', '214591', 'Pennsylvania State University-Penn State Erie-Behrend College'),
    ('Penn State Behrend', '214591', 'Pennsylvania State University-Penn State Erie-Behrend College'),
    ('Pennsylvania State University Penn State Great Valley', '214607', 'Pennsylvania State University-Penn State Great Valley'),
    ('Penn State Great Valley', '214607', 'Pennsylvania State University-Penn State Great Valley'),
    ('Pennsylvania State University Penn State New Kensington', '214625', 'Pennsylvania State University-Penn State New Kensington'),
    ('Penn State New Kensington', '214625', 'Pennsylvania State University-Penn State New Kensington'),
    ('Pennsylvania State University Penn State Shenango', '214634', 'Pennsylvania State University-Penn State Shenango'),
    ('Penn State Shenango', '214634', 'Pennsylvania State University-Penn State Shenango'),
    ('Pennsylvania State University Penn State Wilkes Barre', '214643', 'Pennsylvania State University-Penn State Wilkes-Barre'),
    ('Penn State Wilkes Barre', '214643', 'Pennsylvania State University-Penn State Wilkes-Barre'),
    ('Pennsylvania State University Penn State Scranton', '214652', 'Pennsylvania State University-Penn State Scranton'),
    ('Penn State Scranton', '214652', 'Pennsylvania State University-Penn State Scranton'),
    ('Pennsylvania State University Penn State Lehigh Valley', '214670', 'Pennsylvania State University-Penn State Lehigh Valley'),
    ('Penn State Lehigh Valley', '214670', 'Pennsylvania State University-Penn State Lehigh Valley'),
    ('Pennsylvania State University Penn State Altoona', '214689', 'Pennsylvania State University-Penn State Altoona'),
    ('Penn State Altoona', '214689', 'Pennsylvania State University-Penn State Altoona'),
    ('Pennsylvania State University Penn State Beaver', '214698', 'Pennsylvania State University-Penn State Beaver'),
    ('Penn State Beaver', '214698', 'Pennsylvania State University-Penn State Beaver'),
    ('Pennsylvania State University Penn State Berks', '214704', 'Pennsylvania State University-Penn State Berks'),
    ('Penn State Berks', '214704', 'Pennsylvania State University-Penn State Berks'),
    ('Pennsylvania State University Penn State Harrisburg', '214713', 'Pennsylvania State University-Penn State Harrisburg'),
    ('Penn State Harrisburg', '214713', 'Pennsylvania State University-Penn State Harrisburg'),
    ('Pennsylvania State University Penn State Brandywine', '214731', 'Pennsylvania State University-Penn State Brandywine'),
    ('Penn State Brandywine', '214731', 'Pennsylvania State University-Penn State Brandywine'),
    ('Pennsylvania State University Penn State DuBois', '214740', 'Pennsylvania State University-Penn State DuBois'),
    ('Penn State DuBois', '214740', 'Pennsylvania State University-Penn State DuBois'),
    ('Pennsylvania State University Penn State Fayette Eberly', '214759', 'Pennsylvania State University-Penn State Fayette- Eberly'),
    ('Penn State Fayette Eberly', '214759', 'Pennsylvania State University-Penn State Fayette- Eberly'),
    ('Pennsylvania State University Penn State Hazleton', '214768', 'Pennsylvania State University-Penn State Hazleton'),
    ('Penn State Hazleton', '214768', 'Pennsylvania State University-Penn State Hazleton'),
    ('Pennsylvania State University Penn State Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Pennsylvania State University - Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State University Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State University Greater Allegheny Campus', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State University Penn State Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Pennsylvania State University Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Pennsylvania State University Greater Allegheny Campus', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('The Pennsylvania State University Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('The Pennsylvania State University Greater Allegheny Campus', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State Greater Allegheny Campus', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State University McKeesport PA', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State University McKeesport Greater Allegheny', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Penn State Greater Allegheny Formerly McKeesport Branch Campus', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
    ('Pennsylvania State University Penn State Mont Alto', '214795', 'Pennsylvania State University-Penn State Mont Alto'),
    ('Penn State Mont Alto', '214795', 'Pennsylvania State University-Penn State Mont Alto'),
    ('Pennsylvania State University Penn State Abington', '214801', 'Pennsylvania State University-Penn State Abington'),
    ('Penn State Abington', '214801', 'Pennsylvania State University-Penn State Abington'),
    ('Pennsylvania State University Penn State Schuylkill', '214810', 'Pennsylvania State University-Penn State Schuylkill'),
    ('Penn State Schuylkill', '214810', 'Pennsylvania State University-Penn State Schuylkill'),
    ('Pennsylvania State University Penn State York', '214829', 'Pennsylvania State University-Penn State York'),
    ('Penn State York', '214829', 'Pennsylvania State University-Penn State York'),
    ('Pennsylvania State University World Campus', '479956', 'Pennsylvania State University-World Campus'),
    ('Penn State World Campus', '479956', 'Pennsylvania State University-World Campus'),
]

def _normalize_school_name_value(value):
    if pd.isna(value):
        return ''
    return ' '.join(str(value).lower().replace('&', ' and ').replace('-', ' ').replace(',', ' ').split())

PROMINENT_SCHOOL_ENTITY_UNITID_OVERRIDES = [
    ('parent_rsid', '-1131974', '104151', 'Arizona State University Campus Immersion'),
    ('ultimate_parent_rsid', '33577', '104151', 'Arizona State University Campus Immersion'),
    ('parent_rsid', '-922326', '110635', 'University of California-Berkeley'),
    ('ultimate_parent_rsid', '150461', '110635', 'University of California-Berkeley'),
    ('parent_rsid', '-744165', '228778', 'The University of Texas at Austin'),
    ('ultimate_parent_rsid', '121324', '228778', 'The University of Texas at Austin'),
    ('parent_rsid', '-1246443', '163268', 'University of Maryland-Baltimore County'),
    ('ultimate_parent_rsid', '153328', '163268', 'University of Maryland-Baltimore County'),
    ('parent_rsid', '-1116447', '179867', 'Washington University in St Louis'),
    ('ultimate_parent_rsid', '160755', '179867', 'Washington University in St Louis'),
    ('ultimate_parent_rsid', '38858', '110617', 'California State University-Sacramento'),
    ('parent_rsid', '-1207988', '234979', 'Columbia Basin College'),
    ('ultimate_parent_rsid', '194285', '234979', 'Columbia Basin College'),
    # Branch-level Penn State entity. Do not map parent 120376 here; that is Penn State Main/system.
    ('rsid', '120396', '214786', 'Pennsylvania State University-Penn State Greater Allegheny'),
]

BLOCKED_SCHOOL_ENTITY_UNITID_OVERRIDES = {
    # This entity has shown up as James Madison / Madison University in raw data, not Madison Area Technical College.
    ('ultimate_parent_rsid', '1476', '238263'),
    ('parent_rsid', '-4390029', '238263'),
    ('rsid', '1476', '238263'),
}

def load_institution_entity_mapping():
    for path in INSTITUTION_MAPPING_PATHS:
        if not Path(path).exists():
            continue
        raw = pd.read_csv(path, dtype=str)
        lower = {c.lower(): c for c in raw.columns}
        if {'revelio_id_type', 'rcid', 'unitid', 'ipeds_name'}.issubset(lower):
            out = raw[[lower['revelio_id_type'], lower['rcid'], lower['unitid'], lower['ipeds_name']]].copy()
            out.columns = ['entity_type', 'entity_rsid', 'unitid', 'ipeds_name']
        elif {'entity_type', 'entity_rsid', 'unitid', 'ipeds_name'}.issubset(lower):
            out = raw[[lower['entity_type'], lower['entity_rsid'], lower['unitid'], lower['ipeds_name']]].copy()
            out.columns = ['entity_type', 'entity_rsid', 'unitid', 'ipeds_name']
        else:
            print(f'Institution mapping skipped; {path} does not have recognized columns.')
            continue
        out['entity_type'] = out['entity_type'].fillna('').astype(str).str.strip().str.lower()
        out['entity_rsid'] = out['entity_rsid'].fillna('').astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        out['unitid'] = out['unitid'].fillna('').astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        out['ipeds_name'] = out['ipeds_name'].fillna('').astype(str).str.strip()
        out = out[
            out['entity_type'].isin(['parent_rsid', 'ultimate_parent_rsid', 'rsid'])
            & out['entity_rsid'].ne('')
            & out['unitid'].ne('')
            & out['ipeds_name'].ne('')
        ].copy()
        if BLOCKED_SCHOOL_ENTITY_UNITID_OVERRIDES:
            blocked = set(BLOCKED_SCHOOL_ENTITY_UNITID_OVERRIDES)
            keep_mask = ~out.apply(lambda r: (r['entity_type'], r['entity_rsid'], r['unitid']) in blocked, axis=1)
            dropped = int((~keep_mask).sum())
            if dropped:
                print(f'Dropped {dropped:,} blocked institution entity mapping row(s).')
            out = out[keep_mask].copy()
        out = out.drop_duplicates(['entity_type', 'entity_rsid'], keep='first').reset_index(drop=True)
        print(f'Loaded {len(out):,} institution entity mapping rows from {path}')
        return out
    print('No Revelio/IPEDS institution mapping file found; school entity unitid repairs skipped.')
    return pd.DataFrame(columns=['entity_type', 'entity_rsid', 'unitid', 'ipeds_name'])

institution_entity_mapping = load_institution_entity_mapping()
entity_override_df = pd.DataFrame(PROMINENT_SCHOOL_ENTITY_UNITID_OVERRIDES, columns=['entity_type', 'entity_rsid', 'unitid', 'ipeds_name'])
if len(entity_override_df):
    institution_entity_mapping = pd.concat([entity_override_df, institution_entity_mapping], ignore_index=True)
    institution_entity_mapping = institution_entity_mapping.drop_duplicates(['entity_type', 'entity_rsid'], keep='first').reset_index(drop=True)
institution_name_mapping = pd.DataFrame(PROMINENT_SCHOOL_NAME_UNITID_OVERRIDES, columns=['school_name', 'unitid', 'ipeds_name'])
institution_name_mapping['school_name_norm'] = institution_name_mapping['school_name'].map(_normalize_school_name_value)
institution_name_mapping = institution_name_mapping[institution_name_mapping['school_name_norm'].ne('')]
institution_name_mapping = institution_name_mapping.drop_duplicates('school_name_norm', keep='first')

entity_join_specs = [
    ('parent_rsid', ['parent_rsid', 'school_parent_rsid', 'education_parent_rsid']),
    ('ultimate_parent_rsid', ['ultimate_parent_rsid', 'school_ultimate_parent_rsid', 'education_ultimate_parent_rsid']),
    ('rsid', ['rsid', 'school_rsid', 'education_rsid']),
]
entity_join_clauses = []
entity_unit_exprs = []
entity_name_exprs = []
for idx, (entity_type, candidates) in enumerate(entity_join_specs):
    col = next((candidate for candidate in candidates if candidate.lower() in EDUCATION_COLUMNS), None)
    if not col:
        continue
    alias = f'm{idx}'
    entity_join_clauses.append(
        f"LEFT JOIN {SCRATCH}.SCHOOL_ENTITY_UNITID_MAPPING {alias} "
        f"ON {alias}.entity_type = '{entity_type}' "
        f"AND TO_VARCHAR(e.{col}) = {alias}.entity_rsid"
    )
    entity_unit_exprs.append(f'{alias}.unitid')
    entity_name_exprs.append(f'{alias}.ipeds_name')

school_name_candidates = [
    'ipeds_name', 'school_name', 'institution_name', 'education_school_name',
    'revelio_school_name', 'entity_name', 'university_name', 'name',
    'school', 'institution', 'revelio_name', 'raw_school_name',
    'final_school_name_raw', 'school_norm', 'school_name_raw', 'education_school_name_raw',
]
available_school_name_norm_exprs = []
for col in school_name_candidates:
    if col.lower() not in EDUCATION_COLUMNS:
        continue
    raw_name_expr = f"NULLIF(TO_VARCHAR(e.{col}), '')"
    norm_name_expr = f"LOWER(REGEXP_REPLACE(TRIM(REPLACE(REPLACE({raw_name_expr}, '&', ' and '), ',', ' ')), '[^a-z0-9]+', ' '))"
    available_school_name_norm_exprs.append(norm_name_expr)
if available_school_name_norm_exprs:
    school_name_match_sql = ' OR '.join(f"{expr} = mn.school_name_norm" for expr in available_school_name_norm_exprs)
    name_join_clause = f"LEFT JOIN {SCRATCH}.SCHOOL_NAME_UNITID_OVERRIDES mn ON ({school_name_match_sql})"
else:
    name_join_clause = ''

if len(institution_entity_mapping) or len(institution_name_mapping):
    conn = sfClient.connect()
    cur = conn.cursor()
    cur.execute(f"CREATE OR REPLACE TABLE {SCRATCH}.SCHOOL_ENTITY_UNITID_MAPPING (entity_type VARCHAR, entity_rsid VARCHAR, unitid VARCHAR, ipeds_name VARCHAR)")
    if len(institution_entity_mapping):
        cur.executemany(
            f"INSERT INTO {SCRATCH}.SCHOOL_ENTITY_UNITID_MAPPING (entity_type, entity_rsid, unitid, ipeds_name) VALUES (%s, %s, %s, %s)",
            list(institution_entity_mapping[['entity_type', 'entity_rsid', 'unitid', 'ipeds_name']].itertuples(index=False, name=None)),
        )
    cur.execute(f"CREATE OR REPLACE TABLE {SCRATCH}.SCHOOL_NAME_UNITID_OVERRIDES (school_name_norm VARCHAR, unitid VARCHAR, ipeds_name VARCHAR)")
    if len(institution_name_mapping):
        cur.executemany(
            f"INSERT INTO {SCRATCH}.SCHOOL_NAME_UNITID_OVERRIDES (school_name_norm, unitid, ipeds_name) VALUES (%s, %s, %s)",
            list(institution_name_mapping[['school_name_norm', 'unitid', 'ipeds_name']].itertuples(index=False, name=None)),
        )

    # Branch-specific raw school names should beat broad entity mappings such as
    # "Penn State University" -> Main Campus. Generic rows still fall through.
    unitid_sources = (['mn.unitid'] if name_join_clause else []) + entity_unit_exprs + ['TO_VARCHAR(e.unitid)']
    name_sources = (['mn.ipeds_name'] if name_join_clause else []) + entity_name_exprs + ['TO_VARCHAR(e.ipeds_name)']
    mapped_flag_terms = [f'{expr} IS NOT NULL' for expr in (['mn.unitid'] if name_join_clause else []) + entity_unit_exprs]
    mapped_flag_sql = ' OR '.join(mapped_flag_terms) if mapped_flag_terms else 'FALSE'
    joins_sql = '\n        '.join(entity_join_clauses + ([name_join_clause] if name_join_clause else []))
    mapped_education_table = f'{SCRATCH}.EDUCATION_WITH_CIP_SCHOOL_MAPPED'
    cur.execute(f"""
        CREATE OR REPLACE TABLE {mapped_education_table} AS
        SELECT
            e.* EXCLUDE (unitid, ipeds_name),
            COALESCE({', '.join(unitid_sources)}) AS unitid,
            COALESCE({', '.join(name_sources)}) AS ipeds_name,
            CASE WHEN {mapped_flag_sql} THEN 1 ELSE 0 END AS institution_mapping_override_flag
        FROM {EDUCATION_CIP} e
        {joins_sql}
    """)
    EDUCATION_CIP = mapped_education_table
    education_schema_probe = sfClient.load_df(f"SELECT * FROM {EDUCATION_CIP} LIMIT 0")
    EDUCATION_COLUMNS = {c.lower() for c in education_schema_probe.columns}
    print(f'Applied institution UnitID repairs through {mapped_education_table}.')
    prominent_unitids = [
        '104151','110617','110635','110644','110653','110662','445188','110671','110680','110705','110714',
        '129367','163268','179867','221643','222576','228778','234979','238263',
        '214591','214607','214625','214634','214643','214652','214670','214689','214698','214704','214713',
        '214731','214740','214759','214768','214777','214786','214795','214801','214810','214829','479956',
    ]
    prominent_sql = ','.join(f"'{u}'" for u in prominent_unitids)
    prominent_counts = sfClient.load_df(f"""
        SELECT CAST(unitid AS VARCHAR) AS unitid, MAX(ipeds_name) AS ipeds_name,
               COUNT(*) AS education_rows, COUNT(DISTINCT user_id) AS users
        FROM {EDUCATION_CIP}
        WHERE CAST(unitid AS VARCHAR) IN ({prominent_sql})
        GROUP BY CAST(unitid AS VARCHAR)
        ORDER BY CAST(unitid AS VARCHAR)
    """)
    print('Prominent school mapping counts after repair:')
    display(prominent_counts)
else:
    print('Institution mapping repair skipped: no mapping rows and no name overrides available.')

BERKELEY_COMPETITOR_SCHOOL_NAMES = [
    'University of California-Berkeley',
    'Stanford University',
    'University of California-Los Angeles',
    'Carnegie Mellon University',
    'Massachusetts Institute of Technology',
    'University of Michigan-Ann Arbor',
    'Georgia Institute of Technology-Main Campus',
    'Columbia University in the City of New York',
    'Barnard College',
    'Teachers College at Columbia University',
    'Brown University',
    'Harvard University',
    'Yale University',
    'Princeton University',
    'University of Pennsylvania',
    'Cornell University',
    'Duke University',
    'Northwestern University',
    'University of Chicago',
    'New York University',
    'University of Southern California',
    'University of Washington-Seattle Campus',
    'University of Virginia-Main Campus',
    'University of North Carolina at Chapel Hill',
]

def degree_label_sql(alias='e'):
    alias = alias.strip()
    doctor_cip = degree_cip_sql(alias)
    return f"""
    CASE
        WHEN {alias}.degree = 'Associate' THEN 'Associates'
        WHEN {alias}.degree = 'Bachelor' THEN 'Bachelors'
        WHEN {alias}.degree IN ('Master', 'MBA') THEN 'Masters'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('17') THEN 'Research Doctorate'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('18', '09', '9', '10', '11') THEN 'Professional Doctorate'
        WHEN {alias}.degree = 'Doctor' AND LEFT(COALESCE({doctor_cip}, ''), 2) = '22' THEN 'Professional Doctorate'
        WHEN {alias}.degree = 'Doctor' AND LEFT(COALESCE({doctor_cip}, ''), 5) = '51.12' THEN 'Professional Doctorate'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('19') THEN 'Other Doctorate'
        WHEN {alias}.degree = 'Doctor' THEN 'Research Doctorate'
        ELSE {alias}.degree
    END
    """.strip()


def postgrad_degree_label_sql(alias='e'):
    alias = alias.strip()
    doctor_cip = degree_cip_sql(alias)
    return f"""
    CASE
        WHEN {alias}.degree = 'Master' THEN 'Masters'
        WHEN {alias}.degree = 'MBA' THEN 'MBA'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('17') THEN 'PhD'
        WHEN {alias}.degree = 'Doctor' AND LEFT(COALESCE({doctor_cip}, ''), 2) = '22' THEN 'LAW'
        WHEN {alias}.degree = 'Doctor' AND LEFT(COALESCE({doctor_cip}, ''), 5) = '51.12' THEN 'MD'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('18', '09', '9', '10', '11') THEN 'Professional Doctorate'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('19') THEN 'Other Doctorate'
        WHEN {alias}.degree = 'Doctor' THEN 'Doctorate'
        ELSE {alias}.degree
    END
    """.strip()

def ipeds_degree_level_sql(alias='e'):
    alias = alias.strip()
    return f"""
    CASE
        WHEN {alias}.degree = 'Associate' THEN 'Associates'
        WHEN {alias}.degree = 'Bachelor' THEN 'Bachelors'
        WHEN {alias}.degree IN ('Master', 'MBA') THEN 'Masters'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('17') THEN 'Research Doctorate'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('18', '09', '10', '11') THEN 'Professional Doctorate'
        WHEN {alias}.degree = 'Doctor' AND {alias}.ipeds_awlevel IN ('19') THEN 'Other Doctorate'
        WHEN {alias}.degree = 'Doctor' THEN 'Research Doctorate'
        ELSE {alias}.degree
    END
    """.strip()

# --- CIP code descriptions (2-digit, 4-digit, 6-digit) ---
cip_raw = pd.read_csv(DATA_DIR / 'CIPCode2020.csv', dtype=str, encoding='utf-8-sig')
cip_raw['CIPCode'] = cip_raw['CIPCode'].str.replace(r'^="?|"$', '', regex=True)
cip_raw['CIPFamily'] = cip_raw['CIPFamily'].str.replace(r'^="?|"$', '', regex=True)
cip_raw['CIPTitle'] = cip_raw['CIPTitle'].str.rstrip('.')

cip2_titles = {}
cip4_titles = {}
cip6_titles = {}

for _, row in cip_raw.iterrows():
    code = row['CIPCode'].strip()
    title = row['CIPTitle'].strip() if pd.notna(row['CIPTitle']) else ''
    if len(code) == 2:
        cip2_titles[code] = title
    elif len(code) == 5:
        cip4_titles[code] = title
    elif len(code) == 7:
        cip6_titles[code] = title

print(f'CIP titles loaded: {len(cip2_titles)} families, {len(cip4_titles)} mid-level, {len(cip6_titles)} detailed')

# --- IPEDS completions (for coverage comparison) ---
COMP_DIR = DATA_DIR / '_A'
TOTAL_COL_BY_YEAR = {}
for y in range(2000, 2001):
    TOTAL_COL_BY_YEAR[y] = ('crace15', False)
for y in range(2001, 2003):
    TOTAL_COL_BY_YEAR[y] = ('crace15', True)
for y in range(2003, 2008):
    TOTAL_COL_BY_YEAR[y] = ('crace24', True)
for y in range(2008, 2030):
    TOTAL_COL_BY_YEAR[y] = ('ctotalt', True)


def _read_ipeds_completion_file(path: Path) -> pd.DataFrame:
    year = int(path.name[1:5])
    df = None
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as zf:
            lower_names = {name.lower(): name for name in zf.namelist()}
            preferred = f'c{year}_a.csv'
            member = lower_names.get(preferred)
            if member is None:
                csv_members = [name for name in zf.namelist() if name.lower().endswith('.csv') and '_rv' not in name.lower()]
                if not csv_members:
                    raise ValueError(f'No completion CSV found in {path.name}')
                member = csv_members[0]
            raw = zf.read(member)
        for enc in ['utf-8-sig', 'latin-1']:
            try:
                df = pd.read_csv(io.BytesIO(raw), dtype=str, encoding=enc, low_memory=False)
                break
            except UnicodeDecodeError:
                continue
    else:
        for enc in ['utf-8-sig', 'latin-1']:
            try:
                df = pd.read_csv(path, dtype=str, encoding=enc, low_memory=False)
                break
            except UnicodeDecodeError:
                continue
    if df is None:
        raise ValueError(f'Unable to read {path}')
    df.columns = [c.lower().strip() for c in df.columns]
    total_col, has_awlevel = TOTAL_COL_BY_YEAR.get(year, ('ctotalt', True))
    if total_col not in df.columns:
        return pd.DataFrame(columns=['unitid', 'cipcode', 'awlevel', 'ctotalt', 'year'])
    out = df[['unitid', 'cipcode']].copy()
    if has_awlevel and 'awlevel' in df.columns:
        out['awlevel'] = df['awlevel'].astype(str)
    else:
        out['awlevel'] = 'unknown'
    out['ctotalt'] = pd.to_numeric(df[total_col], errors='coerce').fillna(0).astype(int)
    out['year'] = year
    return out


def get_local_ipeds_comp(comp_dir: Path | None = None):
    comp_dir = Path(comp_dir or COMP_DIR)
    seen_years = set()
    comp_frames = []
    csv_files = sorted(comp_dir.glob('c*_a.csv'))
    for f in csv_files:
        comp_frames.append(_read_ipeds_completion_file(f))
        seen_years.add(int(f.name[1:5]))
    for z in sorted(comp_dir.glob('c*_a.zip')) + sorted(comp_dir.glob('C*_A.zip')):
        year = int(z.name[1:5])
        if year in seen_years:
            continue
        comp_frames.append(_read_ipeds_completion_file(z))
        seen_years.add(year)
    if not comp_frames:
        raise FileNotFoundError(f'No IPEDS completion files found in {comp_dir}')
    comp = pd.concat(comp_frames, ignore_index=True)
    comp['unitid'] = comp['unitid'].astype(str)
    comp['cipcode'] = comp['cipcode'].astype(str).str.replace(r'^="?|"$', '', regex=True)
    comp['awlevel'] = comp['awlevel'].astype(str)
    return comp_dir, comp


_, comp = get_local_ipeds_comp()
print(f'IPEDS completions: {len(comp):,} rows, years {comp["year"].min()}-{comp["year"].max()}')

cip2_df = pd.DataFrame([{'cip2': k, 'cip2_title': v} for k, v in cip2_titles.items()])
cip4_df = pd.DataFrame([{'cip4': k, 'cip4_title': v} for k, v in cip4_titles.items()])
cip6_df = pd.DataFrame([{'cip6': k, 'cip6_title': v} for k, v in cip6_titles.items()])

print(f'\nSample CIP2: {list(cip2_titles.items())[:5]}')
print(f'Sample CIP4: {list(cip4_titles.items())[:5]}')

SALARY_REAL_BASE_YEAR = 2024
SALARY_REAL_BASE_CPI = 313.689


def weighted_aggregate_sql(
    source_sql,
    group_cols,
    count_alias='n',
    salary_count_alias='salary_obs',
    quantile_aliases=None,
    mean_alias='mean_salary',
    include_salary=True,
    include_seniority=True,
    extra_aggs=None,
    having='SUM(COALESCE(analysis_weight, 1.0)) >= 1'
):
    # Build a Snowflake aggregation with weighted counts and exact weighted salary quantiles.
    quantile_aliases = quantile_aliases or {
        0.10: 'p10_salary',
        0.25: 'p25_salary',
        0.50: 'median_salary',
        0.75: 'p75_salary',
        0.90: 'p90_salary',
    }
    extra_aggs = extra_aggs or []
    group_select = ',\n            '.join(group_cols)
    group_by = ', '.join(group_cols)
    join_condition = ' AND '.join([f'EQUAL_NULL(c.{col}, q.{col})' for col in group_cols]) or '1=1'
    extra_sql = ''
    if extra_aggs:
        extra_sql = ',\n            ' + ',\n            '.join(extra_aggs)
    seniority_sql = ''
    if include_seniority:
        seniority_sql = ",\n            ROUND(SUM(CASE WHEN seniority IS NOT NULL THEN seniority * COALESCE(analysis_weight, 1.0) ELSE 0 END) / NULLIF(SUM(CASE WHEN seniority IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END), 0), 2) AS avg_seniority"
    if include_salary:
        quantile_select = ',\n            '.join(
            f"MIN(CASE WHEN cum_weight >= total_salary_weight * {q} THEN salary END) AS {alias}"
            for q, alias in quantile_aliases.items()
        )
        quantile_cols = ',\n        ' + ',\n        '.join([f'q.{alias}' for alias in quantile_aliases.values()])
        salary_ctes = f""",
    salary_ranked AS (
        SELECT
            {group_select},
            salary,
            COALESCE(analysis_weight, 1.0) AS salary_weight,
            SUM(COALESCE(analysis_weight, 1.0)) OVER (
                PARTITION BY {group_by}
                ORDER BY salary
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS cum_weight,
            SUM(COALESCE(analysis_weight, 1.0)) OVER (PARTITION BY {group_by}) AS total_salary_weight
        FROM source
        WHERE salary IS NOT NULL
          AND COALESCE(analysis_weight, 1.0) > 0
    ),
    salary_quantiles AS (
        SELECT
            {group_select},
            {quantile_select}
        FROM salary_ranked
        GROUP BY {group_by}
    )"""
        salary_count_sql = f""",
            SUM(CASE WHEN salary IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END) AS weighted_salary_n,
            ROUND(SUM(CASE WHEN salary IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END), 0) AS {salary_count_alias},
            COUNT(salary) AS raw_salary_obs,
            ROUND(SUM(CASE WHEN salary IS NOT NULL THEN salary * COALESCE(analysis_weight, 1.0) ELSE 0 END) / NULLIF(SUM(CASE WHEN salary IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END), 0), 0) AS {mean_alias}"""
        final_join = f"LEFT JOIN salary_quantiles q ON {join_condition}"
    else:
        salary_ctes = ''
        salary_count_sql = ''
        quantile_cols = ''
        final_join = ''
    return f"""
WITH source AS (
    {source_sql.strip().rstrip(';')}
),
counts AS (
    SELECT
            {group_select},
            SUM(COALESCE(analysis_weight, 1.0)) AS weighted_n,
            ROUND(SUM(COALESCE(analysis_weight, 1.0)), 0) AS {count_alias},
            COUNT(*) AS raw_n{salary_count_sql}{seniority_sql}{extra_sql}
    FROM source
    GROUP BY {group_by}
    HAVING {having}
){salary_ctes}
SELECT
        c.*{quantile_cols}
FROM counts c
{final_join}
""".strip()

IPEDS_COMPLETIONS_TABLE = f'{SCRATCH}.IPEDS_COMPLETIONS'

print('\nReady')


In [ ]:
# =============================================================
# CELL 1: Define the broad NACE school sample and resolve unitids
# =============================================================
# This uses the broad NACE readiness file when available. It keeps every
# high-confidence school in the accepted data-quality tiers instead of
# truncating to the old 70-school demo list.

NACE_MAX_SCHOOLS = None  # Set to an integer for a smaller test run.
NACE_MIN_READINESS_SCORE = 75.0
NACE_SELECTION_TIERS = {'excellent', 'strong'}
NACE_MATCH_STATUSES = {'high_confidence'}
NACE_RECOMMENDED_UNITIDS = [
    '232423',
    '217819',
    '110714',
    '366711',
    '110671',
    '199218',
    '132903',
    '230038',
    '139931',
    '153603',
    '110565',
    '209542',
    '229027',
    '236939',
    '164076',
    '231174',
    '228459',
    '136172',
    '176017',
    '100751',
    '100858',
    '228875',
    '196079',
    '223232',
    '433660',
    '122409',
    '139959',
    '209551',
    '110680',
    '134097',
    '110653',
    '110617',
    '130943',
    '142115',
    '220978',
    '229115',
    '218663',
    '199148',
    '204796',
    '110644',
    '228723',
    '110608',
    '133951',
    '171100',
    '126818',
    '155399',
    '238032',
    '207388',
    '232982',
    '163268',
    '129020',
    '234030',
    '104179',
    '216339',
    '186380',
    '159391',
    '195003',
    '243780',
    '240444',
    '204857',
    '151351',
    '153658',
    '185590',
    '228778',
    '196097',
    '155317',
    '225511',
    '199120',
    '230764',
    '134130',
]

# Keep the demo/elite peer set available even when the broad NACE
# selection is used. Duplicates are removed after appending.
ELITE_DEMO_UNITIDS = [
    '110635',  # University of California-Berkeley
    '243744',  # Stanford University
    '110662',  # University of California-Los Angeles
    '211440',  # Carnegie Mellon University
    '166683',  # Massachusetts Institute of Technology
    '170976',  # University of Michigan-Ann Arbor
    '139755',  # Georgia Institute of Technology-Main Campus
    '190150',  # Columbia University in the City of New York
    '189097',  # Barnard College
    '196468',  # Teachers College at Columbia University
    '217156',  # Brown University
    '166027',  # Harvard University
    '130794',  # Yale University
    '186131',  # Princeton University
    '215062',  # University of Pennsylvania
    '190415',  # Cornell University
    '198419',  # Duke University
    '147767',  # Northwestern University
    '144050',  # University of Chicago
    '193900',  # New York University
    '123961',  # University of Southern California
    '236948',  # University of Washington-Seattle Campus
    '234076',  # University of Virginia-Main Campus
    '199120',  # University of North Carolina at Chapel Hill
]

NACE_SELECTION_PATHS = []
if os.environ.get('OUTCOMES_SCHOOL_SELECTION_FILE'):
    NACE_SELECTION_PATHS.append(Path(os.environ['OUTCOMES_SCHOOL_SELECTION_FILE']).expanduser().resolve())
NACE_SELECTION_PATHS.extend([
    DATA_DIR / 'recommended_1400_school_run_list.csv',
    Path('recommended_1400_school_run_list.csv'),
    DATA_DIR / 'nace_june5_4year_503_run_list.csv',
    Path('nace_june5_4year_503_run_list.csv'),
    DATA_DIR / 'nace_school_data_readiness.csv',
    Path('nace_school_data_readiness.csv'),
    DATA_DIR / 'nace_school_candidates_matched.csv',
    Path('nace_school_candidates_matched.csv'),
    DATA_DIR / 'nace_recommended_70_schools.csv',
    Path('nace_recommended_70_schools.csv'),
])


def _limit_nace_selection(df):
    if NACE_MAX_SCHOOLS is None:
        return df
    return df.head(int(NACE_MAX_SCHOOLS)).copy()


def _numeric_col(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce')
    return pd.Series(np.nan, index=df.index)


def _normalize_selection_columns(df):
    df = df.copy()
    df.columns = [c.lower().strip() for c in df.columns]
    if 'unitid' not in df.columns and 'matched_unitid' in df.columns:
        df = df.rename(columns={'matched_unitid': 'unitid'})
    if 'ipeds_name' not in df.columns and 'matched_ipeds_name' in df.columns:
        df = df.rename(columns={'matched_ipeds_name': 'ipeds_name'})
    if 'city' not in df.columns and 'matched_city' in df.columns:
        df = df.rename(columns={'matched_city': 'city'})
    if 'state' not in df.columns and 'matched_state' in df.columns:
        df = df.rename(columns={'matched_state': 'state'})
    if 'unitid' not in df.columns:
        raise ValueError('selection file has no unitid or matched_unitid column')
    df = df[df['unitid'].notna()].copy()
    df['unitid'] = df['unitid'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    df = df[df['unitid'] != ''].copy()
    return df


def _filter_readiness_selection(df):
    force_col = next((c for c in ['force_include_for_june5_503_run', 'force_include'] if c in df.columns), None)
    if force_col and df[force_col].astype(str).str.lower().isin({'yes', 'true', '1'}).any():
        df = df[df[force_col].astype(str).str.lower().isin({'yes', 'true', '1'})].copy()
        if 'run_rank' in df.columns:
            df['run_rank'] = pd.to_numeric(df['run_rank'], errors='coerce')
            df = df.sort_values('run_rank').copy()
        return df

    if {'match_status', 'selection_tier', 'data_readiness_score'}.issubset(df.columns):
        readiness = _numeric_col(df, 'data_readiness_score')
        df = df[
            df['match_status'].isin(NACE_MATCH_STATUSES)
            & df['selection_tier'].isin(NACE_SELECTION_TIERS)
            & readiness.ge(NACE_MIN_READINESS_SCORE)
        ].copy()
        df['data_readiness_score'] = readiness.loc[df.index]
        sort_cols = [c for c in ['selection_tier_rank', 'data_readiness_score', 'nace_attendees'] if c in df.columns]
        if 'selection_tier_rank' in sort_cols:
            df['selection_tier_rank'] = _numeric_col(df, 'selection_tier_rank')
        if 'nace_attendees' in sort_cols:
            df['nace_attendees'] = _numeric_col(df, 'nace_attendees')
        if sort_cols:
            ascending = [True if c == 'selection_tier_rank' else False for c in sort_cols]
            df = df.sort_values(sort_cols, ascending=ascending).copy()
        return df

    if 'recommendation' in df.columns and df['recommendation'].eq('include_top70_data_quality').any():
        return df[df['recommendation'].eq('include_top70_data_quality')].copy()

    if 'recommended_rank' in df.columns:
        df['recommended_rank'] = _numeric_col(df, 'recommended_rank')
        return df.sort_values('recommended_rank').copy()

    if 'match_status' in df.columns:
        df = df[df['match_status'].isin(NACE_MATCH_STATUSES)].copy()
    sort_cols = [c for c in ['nace_attendees', 'match_score'] if c in df.columns]
    for col in sort_cols:
        df[col] = _numeric_col(df, col)
    return df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).copy() if sort_cols else df


def _load_nace_selection():
    for path in NACE_SELECTION_PATHS:
        if not path.exists():
            continue
        raw = pd.read_csv(path, dtype=str)
        df = _normalize_selection_columns(raw)
        before = len(df)
        df = _filter_readiness_selection(df)
        df = df.drop_duplicates('unitid').copy()
        df = _limit_nace_selection(df)
        if len(df):
            print(f'Loaded {len(df)} NACE school selections from {path} ({before} matched rows before quality filters)')
            return df, path
        print(f'No schools passed quality filters in {path}; trying next selection file.')
    return None, None


selection_df, selection_source = _load_nace_selection()
if selection_df is not None and len(selection_df):
    target_unitids = selection_df['unitid'].astype(str).tolist()
else:
    print('WARNING: no broad NACE readiness file was found; using embedded 70-school fallback list.')
    target_unitids = list(NACE_RECOMMENDED_UNITIDS)
    selection_df = pd.DataFrame({'unitid': target_unitids, 'recommended_rank': range(1, len(target_unitids) + 1)})
    selection_source = 'embedded NACE_RECOMMENDED_UNITIDS'


def _with_elite_demo_addons(selection_df, target_unitids, selection_source):
    base_unitids = list(dict.fromkeys(str(u) for u in target_unitids if pd.notna(u)))
    force_col = next((c for c in ['force_include_for_june5_503_run', 'force_include'] if c in selection_df.columns), None)
    if force_col and selection_df[force_col].astype(str).str.lower().isin({'yes', 'true', '1'}).any():
        source = f'{selection_source} (explicit force-include list; elite/demo add-ons not appended)'
        return selection_df.copy(), base_unitids, source

    addon_unitids = [u for u in ELITE_DEMO_UNITIDS if u not in base_unitids]
    merged_unitids = base_unitids + addon_unitids
    addon_start_rank = len(base_unitids) + 1

    addon_df = pd.DataFrame({
        'unitid': ELITE_DEMO_UNITIDS,
        'recommended_rank': range(addon_start_rank, addon_start_rank + len(ELITE_DEMO_UNITIDS)),
        'selection_tier': 'elite_demo_addon',
        'match_status': 'manual_addon',
    })
    selection_df = selection_df.copy() if selection_df is not None else pd.DataFrame(columns=['unitid'])
    selection_df['unitid'] = selection_df['unitid'].astype(str).str.replace(r'\.0$', '', regex=True)
    selection_df = pd.concat([selection_df, addon_df], ignore_index=True).drop_duplicates('unitid', keep='first')

    already_included = len(ELITE_DEMO_UNITIDS) - len(addon_unitids)
    source = f'{selection_source} + elite/demo add-ons ({len(addon_unitids)} new, {already_included} already included)'
    return selection_df, merged_unitids, source


selection_df, target_unitids, selection_source = _with_elite_demo_addons(selection_df, target_unitids, selection_source)

# Pull school names and unitids from the same Revelio/IPEDS crosswalk used by the facts.
school_lookup = sfClient.load_df(f"""
    SELECT DISTINCT CAST(unitid AS VARCHAR) AS unitid, ipeds_name
    FROM {EDUCATION_CIP}
    WHERE unitid IS NOT NULL AND ipeds_name IS NOT NULL
""")
school_lookup.columns = [c.lower() for c in school_lookup.columns]
school_lookup['unitid'] = school_lookup['unitid'].astype(str).str.replace(r'\.0$', '', regex=True)
school_lookup['lookup_priority'] = 0

def _load_school_name_fallbacks():
    frames = []
    for path in [
        DATA_DIR / 'recommended_1400_school_run_list.csv',
        Path('recommended_1400_school_run_list.csv'),
        DATA_DIR / 'nace_school_data_readiness.csv',
        Path('nace_school_data_readiness.csv'),
        DATA_DIR / 'nace_school_candidates_matched.csv',
        Path('nace_school_candidates_matched.csv'),
    ]:
        if not path.exists():
            continue
        df = pd.read_csv(path, dtype=str)
        df.columns = [c.lower().strip() for c in df.columns]
        if 'unitid' not in df.columns and 'matched_unitid' in df.columns:
            df = df.rename(columns={'matched_unitid': 'unitid'})
        if 'unitid' not in df.columns:
            continue
        name_col = next((c for c in ['ipeds_name', 'ipeds_name_candidate', 'matched_ipeds_name', 'school_name', 'institution_name', 'name'] if c in df.columns), None)
        if not name_col:
            continue
        out = df[['unitid', name_col]].rename(columns={name_col: 'ipeds_name'}).copy()
        out['lookup_priority'] = 1
        frames.append(out)
        print(f'Loaded fallback school names from {path}')
        break
    manual = pd.DataFrame([
        {'unitid': '110671', 'ipeds_name': 'University of California-Riverside', 'lookup_priority': 2},
        {'unitid': '199148', 'ipeds_name': 'University of North Carolina at Greensboro', 'lookup_priority': 2},
        {'unitid': '133951', 'ipeds_name': 'Florida International University', 'lookup_priority': 2},
        {'unitid': '186380', 'ipeds_name': 'Rutgers University-New Brunswick', 'lookup_priority': 2},
    ])
    frames.append(manual)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['unitid', 'ipeds_name', 'lookup_priority'])

fallback_school_lookup = _load_school_name_fallbacks()
fallback_school_lookup['unitid'] = fallback_school_lookup['unitid'].astype(str).str.replace(r'\.0$', '', regex=True)
fallback_school_lookup['ipeds_name'] = fallback_school_lookup['ipeds_name'].fillna('').astype(str).str.strip()
fallback_school_lookup = fallback_school_lookup[fallback_school_lookup['ipeds_name'] != '']
school_lookup = (
    pd.concat([school_lookup, fallback_school_lookup], ignore_index=True)
    .sort_values(['unitid', 'lookup_priority', 'ipeds_name'])
    .drop_duplicates('unitid', keep='first')
    [['unitid', 'ipeds_name']]
)

order = pd.DataFrame({'unitid': target_unitids, 'selection_rank': range(1, len(target_unitids) + 1)})
selection_cols = [
    c for c in [
        'unitid', 'recommended_rank', 'data_readiness_score', 'selection_tier',
        'match_status', 'nace_attendees', 'nace_names', 'nace_examples', 'city', 'state',
        'ipeds_name', 'run_rank', 'batch_id', 'needs_match_review', 'force_include_for_june5_503_run',
        'total_attendees', 'company_variants'
    ] if c in selection_df.columns
]
selection_meta = selection_df[selection_cols].drop_duplicates('unitid') if selection_cols else pd.DataFrame({'unitid': target_unitids})

school_meta = (
    order
    .merge(selection_meta, on='unitid', how='left')
    .merge(school_lookup, on='unitid', how='left', suffixes=('_selection', ''))
)
if 'ipeds_name_selection' in school_meta.columns:
    school_meta['ipeds_name'] = school_meta['ipeds_name'].fillna(school_meta['ipeds_name_selection'])

missing_rows = school_meta[school_meta['ipeds_name'].isna()].copy()
if len(missing_rows):
    addon_mask = missing_rows.get('selection_tier', pd.Series('', index=missing_rows.index)).eq('elite_demo_addon')
    missing_addons = missing_rows[addon_mask]['unitid'].astype(str).tolist()
    missing_required = missing_rows[~addon_mask]['unitid'].astype(str).tolist()
    if missing_addons:
        print(f'WARNING: skipping elite/demo add-ons missing from {EDUCATION_CIP} lookup: {missing_addons}')
        school_meta = school_meta[~school_meta['unitid'].isin(missing_addons)].copy()
    if missing_required:
        print(f'WARNING: selected UnitIDs missing from all school-name lookups; keeping with unitid labels: {missing_required}')
        missing_required_mask = school_meta['unitid'].isin(missing_required)
        school_meta.loc[missing_required_mask, 'ipeds_name'] = 'Unknown IPEDS unitid ' + school_meta.loc[missing_required_mask, 'unitid'].astype(str)

school_meta = school_meta.drop_duplicates('unitid').reset_index(drop=True)
UNITID_LIST = school_meta['unitid'].astype(str).tolist()
UNITID_SQL = ','.join(f"'{u}'" for u in UNITID_LIST)

print(f'Building broad NACE + elite outcomes bundle for {len(UNITID_LIST)} schools.')
print(f'School selection source: {selection_source}')
print(f'Data-quality filters: match_status in {sorted(NACE_MATCH_STATUSES)}, selection_tier in {sorted(NACE_SELECTION_TIERS)}, readiness >= {NACE_MIN_READINESS_SCORE}')
print(f'Output folder: {OUT_DIR}')
if 'data_readiness_score' in school_meta.columns:
    readiness = pd.to_numeric(school_meta['data_readiness_score'], errors='coerce')
    if readiness.notna().any():
        print(f'Readiness score range: {readiness.min():.1f} - {readiness.max():.1f}')
print('\nSelected schools:')
print(school_meta[['selection_rank', 'unitid', 'ipeds_name']].to_string(index=False))


In [ ]:
# =============================================================
# CELL 1A: U.S. school data-capacity audit
# =============================================================
# Diagnostic only. This does not change UNITID_LIST or the export.
# It estimates how many U.S. bachelor's-granting schools have enough
# Revelio education/current-student/career signal to be plausible additions.

US_CAPACITY_RECENT_YEARS = list(range(2020, min(int(comp['year'].max()), 2025) + 1))
US_CAPACITY_OUTCOME_YEARS = list(range(2015, 2025))
US_CAPACITY_MIN_IPEDS_BACHELORS = 100
RUN_FULL_CAPACITY_AUDIT = os.environ.get('OUTCOMES_RUN_CAPACITY_AUDIT', '1').strip().lower() not in {'0', 'false', 'no'}
CAPACITY_AUDIT_SCHOOL_FILTER_SQL = '' if RUN_FULL_CAPACITY_AUDIT else f'AND CAST(e.unitid AS VARCHAR) IN ({UNITID_SQL})'

print(f'Auditing U.S. school data capacity for IPEDS years {US_CAPACITY_RECENT_YEARS[0]}-{US_CAPACITY_RECENT_YEARS[-1]}...')

# IPEDS is the U.S. school universe. A school enters the audit if it has recent
# bachelor's completions. This intentionally does not require NACE attendance.
ipeds_bach = comp.copy()
ipeds_bach['unitid'] = ipeds_bach['unitid'].astype(str).str.replace(r'\.0$', '', regex=True)
ipeds_bach['awlevel'] = ipeds_bach['awlevel'].astype(str).str.zfill(2)
ipeds_bach['year'] = pd.to_numeric(ipeds_bach['year'], errors='coerce')
ipeds_bach['ctotalt'] = pd.to_numeric(ipeds_bach['ctotalt'], errors='coerce').fillna(0)
ipeds_bach['cipcode'] = ipeds_bach['cipcode'].astype(str).str.replace(r'^="?|"$', '', regex=True)
ipeds_bach = ipeds_bach[
    ipeds_bach['awlevel'].isin({'05'})
    & ipeds_bach['year'].isin(US_CAPACITY_RECENT_YEARS)
    & ipeds_bach['ctotalt'].gt(0)
    & ipeds_bach['cipcode'].str.match(r'^\d{2}\.\d{2}', na=False)
    & ~ipeds_bach['cipcode'].str.startswith('99', na=False)
].copy()
ipeds_bach['cip4'] = ipeds_bach['cipcode'].str[:5]
ipeds_school_capacity = (
    ipeds_bach
    .groupby('unitid', as_index=False)
    .agg(
        recent_ipeds_bachelors=('ctotalt', 'sum'),
        ipeds_bachelor_years=('year', 'nunique'),
        ipeds_bachelor_cip4s=('cip4', 'nunique'),
    )
)
ipeds_school_capacity = ipeds_school_capacity[
    ipeds_school_capacity['recent_ipeds_bachelors'] >= US_CAPACITY_MIN_IPEDS_BACHELORS
].copy()
if not RUN_FULL_CAPACITY_AUDIT:
    ipeds_school_capacity = ipeds_school_capacity[ipeds_school_capacity['unitid'].isin(set(UNITID_LIST))].copy()
    print(f'Compact run: capacity audit restricted to {len(UNITID_LIST)} selected schools.')

current_student_filter_sql = f"""
        e.degree = 'Bachelor'
        AND {assigned_cip4_sql('e')} IS NOT NULL
        AND (
            (e.enddate IS NOT NULL AND YEAR(e.enddate) BETWEEN 2026 AND 2029)
            OR (e.enddate IS NULL AND e.startdate IS NOT NULL AND YEAR(e.startdate) BETWEEN 2022 AND 2025)
        )
""" if 'startdate' in EDUCATION_COLUMNS else f"""
        e.degree = 'Bachelor'
        AND {assigned_cip4_sql('e')} IS NOT NULL
        AND e.enddate IS NOT NULL
        AND YEAR(e.enddate) BETWEEN 2026 AND 2029
"""

capacity_sql = f"""
WITH recent_bachelors AS (
    SELECT DISTINCT
        e.user_id,
        CAST(e.unitid AS VARCHAR) AS unitid,
        e.ipeds_name,
        YEAR(e.enddate) AS grad_year,
        {assigned_cip4_sql('e')} AS cip4,
        TRY_TO_DOUBLE(e.cip_probability) AS cip_probability
    FROM {EDUCATION_CIP} e
    WHERE e.unitid IS NOT NULL
      {CAPACITY_AUDIT_SCHOOL_FILTER_SQL}
      AND e.degree = 'Bachelor'
      AND e.enddate IS NOT NULL
      AND YEAR(e.enddate) BETWEEN {US_CAPACITY_RECENT_YEARS[0]} AND {US_CAPACITY_RECENT_YEARS[-1]}
), recent_by_school AS (
    SELECT
        unitid,
        ANY_VALUE(ipeds_name) AS ipeds_name,
        COUNT(DISTINCT user_id) AS recent_bachelor_profiles,
        COUNT(DISTINCT CASE WHEN cip4 IS NOT NULL THEN user_id END) AS recent_cip4_profiles,
        COUNT(DISTINCT CASE WHEN cip4 IS NOT NULL AND COALESCE(cip_probability, 0) >= 0.8 THEN user_id END) AS recent_high_conf_cip4_profiles,
        COUNT(DISTINCT cip4) AS observed_cip4s
    FROM recent_bachelors
    GROUP BY unitid
), major_breadth AS (
    SELECT unitid, COUNT(*) AS cip4s_with_10_profiles
    FROM (
        SELECT unitid, cip4, COUNT(DISTINCT user_id) AS profiles
        FROM recent_bachelors
        WHERE cip4 IS NOT NULL
        GROUP BY unitid, cip4
        HAVING COUNT(DISTINCT user_id) >= 10
    )
    GROUP BY unitid
), outcome_bachelors AS (
    SELECT DISTINCT
        e.user_id,
        CAST(e.unitid AS VARCHAR) AS unitid,
        e.enddate AS grad_date,
        {assigned_cip4_sql('e')} AS cip4
    FROM {EDUCATION_CIP} e
    WHERE e.unitid IS NOT NULL
      {CAPACITY_AUDIT_SCHOOL_FILTER_SQL}
      AND e.degree = 'Bachelor'
      AND e.enddate IS NOT NULL
      AND YEAR(e.enddate) BETWEEN {US_CAPACITY_OUTCOME_YEARS[0]} AND {US_CAPACITY_OUTCOME_YEARS[-1]}
), one_year_positions AS (
    SELECT
        b.unitid,
        COUNT(DISTINCT b.user_id) AS one_year_position_profiles
    FROM outcome_bachelors b
    JOIN {POSITION_TABLE} p
      ON b.user_id = p.user_id
     AND p.startdate <= DATEADD('year', 1, b.grad_date)
     AND (p.enddate >= DATEADD('year', 1, b.grad_date) OR p.enddate IS NULL)
    GROUP BY b.unitid
), outcome_by_school AS (
    SELECT
        unitid,
        COUNT(DISTINCT user_id) AS outcome_bachelor_profiles,
        COUNT(DISTINCT CASE WHEN cip4 IS NOT NULL THEN user_id END) AS outcome_cip4_profiles
    FROM outcome_bachelors
    GROUP BY unitid
), current_students AS (
    SELECT DISTINCT
        e.user_id,
        CAST(e.unitid AS VARCHAR) AS unitid,
        {assigned_cip4_sql('e')} AS cip4
    FROM {EDUCATION_CIP} e
    WHERE 1=1
      {CAPACITY_AUDIT_SCHOOL_FILTER_SQL}
      AND {current_student_filter_sql}
), current_by_school AS (
    SELECT
        unitid,
        COUNT(DISTINCT user_id) AS current_bachelor_profiles,
        COUNT(DISTINCT CASE WHEN cip4 IS NOT NULL THEN user_id END) AS current_cip4_profiles,
        COUNT(DISTINCT cip4) AS current_cip4s
    FROM current_students
    GROUP BY unitid
)
SELECT
    r.unitid,
    r.ipeds_name,
    r.recent_bachelor_profiles,
    r.recent_cip4_profiles,
    r.recent_high_conf_cip4_profiles,
    r.observed_cip4s,
    COALESCE(m.cip4s_with_10_profiles, 0) AS cip4s_with_10_profiles,
    COALESCE(o.outcome_bachelor_profiles, 0) AS outcome_bachelor_profiles,
    COALESCE(o.outcome_cip4_profiles, 0) AS outcome_cip4_profiles,
    COALESCE(y.one_year_position_profiles, 0) AS one_year_position_profiles,
    COALESCE(c.current_bachelor_profiles, 0) AS current_bachelor_profiles,
    COALESCE(c.current_cip4_profiles, 0) AS current_cip4_profiles,
    COALESCE(c.current_cip4s, 0) AS current_cip4s
FROM recent_by_school r
LEFT JOIN major_breadth m USING (unitid)
LEFT JOIN outcome_by_school o USING (unitid)
LEFT JOIN one_year_positions y USING (unitid)
LEFT JOIN current_by_school c USING (unitid)
"""

observed_capacity = sfClient.load_df(capacity_sql)
observed_capacity.columns = [c.lower() for c in observed_capacity.columns]
observed_capacity['unitid'] = observed_capacity['unitid'].astype(str).str.replace(r'\.0$', '', regex=True)

us_school_capacity = ipeds_school_capacity.merge(observed_capacity, on='unitid', how='left')
for col in [
    'recent_bachelor_profiles', 'recent_cip4_profiles', 'recent_high_conf_cip4_profiles',
    'observed_cip4s', 'cip4s_with_10_profiles', 'outcome_bachelor_profiles',
    'outcome_cip4_profiles', 'one_year_position_profiles', 'current_bachelor_profiles',
    'current_cip4_profiles', 'current_cip4s'
]:
    us_school_capacity[col] = pd.to_numeric(us_school_capacity[col], errors='coerce').fillna(0)

us_school_capacity['cip4_profile_rate_pct'] = 100 * us_school_capacity['recent_cip4_profiles'] / us_school_capacity['recent_bachelor_profiles'].replace(0, np.nan)
us_school_capacity['high_conf_cip4_rate_pct'] = 100 * us_school_capacity['recent_high_conf_cip4_profiles'] / us_school_capacity['recent_bachelor_profiles'].replace(0, np.nan)
us_school_capacity['revelio_to_ipeds_profile_rate_pct'] = 100 * us_school_capacity['recent_bachelor_profiles'] / us_school_capacity['recent_ipeds_bachelors'].replace(0, np.nan)
us_school_capacity['one_year_position_rate_pct'] = 100 * us_school_capacity['one_year_position_profiles'] / us_school_capacity['outcome_bachelor_profiles'].replace(0, np.nan)
us_school_capacity['current_to_recent_profile_rate_pct'] = 100 * us_school_capacity['current_cip4_profiles'] / us_school_capacity['recent_cip4_profiles'].replace(0, np.nan)

score = (
    25 * np.minimum(us_school_capacity['recent_cip4_profiles'] / 750, 1)
    + 20 * np.minimum(us_school_capacity['cip4_profile_rate_pct'].fillna(0) / 75, 1)
    + 15 * np.minimum(us_school_capacity['cip4s_with_10_profiles'] / 12, 1)
    + 25 * np.minimum(us_school_capacity['one_year_position_rate_pct'].fillna(0) / 45, 1)
    + 15 * np.minimum(us_school_capacity['current_cip4_profiles'] / 250, 1)
)
us_school_capacity['data_capacity_score'] = score.round(1)

conditions = [
    (us_school_capacity['recent_cip4_profiles'] >= 1000)
    & (us_school_capacity['cip4_profile_rate_pct'] >= 70)
    & (us_school_capacity['cip4s_with_10_profiles'] >= 12)
    & (us_school_capacity['one_year_position_rate_pct'] >= 45),
    (us_school_capacity['recent_cip4_profiles'] >= 500)
    & (us_school_capacity['cip4_profile_rate_pct'] >= 60)
    & (us_school_capacity['cip4s_with_10_profiles'] >= 8)
    & (us_school_capacity['one_year_position_rate_pct'] >= 35),
    (us_school_capacity['recent_cip4_profiles'] >= 250)
    & (us_school_capacity['cip4_profile_rate_pct'] >= 50)
    & (us_school_capacity['cip4s_with_10_profiles'] >= 5)
    & (us_school_capacity['one_year_position_rate_pct'] >= 25),
]
us_school_capacity['data_capacity_tier'] = np.select(conditions, ['excellent', 'strong', 'usable'], default='watchlist')
us_school_capacity['currently_selected'] = us_school_capacity['unitid'].isin(set(UNITID_LIST))

ordered_cols = [
    'unitid', 'ipeds_name', 'data_capacity_tier', 'data_capacity_score', 'currently_selected',
    'recent_ipeds_bachelors', 'recent_bachelor_profiles', 'recent_cip4_profiles',
    'cip4_profile_rate_pct', 'revelio_to_ipeds_profile_rate_pct', 'cip4s_with_10_profiles',
    'one_year_position_rate_pct', 'current_cip4_profiles', 'current_to_recent_profile_rate_pct',
    'ipeds_bachelor_cip4s', 'observed_cip4s', 'current_cip4s', 'outcome_bachelor_profiles',
    'one_year_position_profiles'
]
us_school_capacity = us_school_capacity.sort_values(
    ['data_capacity_score', 'recent_cip4_profiles', 'one_year_position_rate_pct'],
    ascending=[False, False, False],
).reset_index(drop=True)

capacity_path = OUT_DIR / 'us_school_data_capacity_audit.csv'
us_school_capacity[ordered_cols].to_csv(capacity_path, index=False)

# Rerunnable, lightweight quality-bin summary. Use this before committing to a
# full rebuild to see how many schools are excellent/strong/usable/watchlist
# under the current thresholds and source tables.
quality_bin_order = ['excellent', 'strong', 'usable', 'watchlist']
quality_bin_counts = (
    us_school_capacity
    .assign(currently_selected=us_school_capacity['currently_selected'].fillna(False).astype(bool))
    .groupby('data_capacity_tier', dropna=False)
    .agg(
        schools=('unitid', 'nunique'),
        currently_selected=('currently_selected', 'sum'),
        avg_capacity_score=('data_capacity_score', 'mean'),
        recent_ipeds_bachelors=('recent_ipeds_bachelors', 'sum'),
        recent_bachelor_profiles=('recent_bachelor_profiles', 'sum'),
        recent_cip4_profiles=('recent_cip4_profiles', 'sum'),
        current_cip4_profiles=('current_cip4_profiles', 'sum'),
    )
    .reindex(quality_bin_order)
    .fillna({
        'schools': 0,
        'currently_selected': 0,
        'recent_ipeds_bachelors': 0,
        'recent_bachelor_profiles': 0,
        'recent_cip4_profiles': 0,
        'current_cip4_profiles': 0,
    })
    .reset_index()
)
quality_bin_counts['schools'] = quality_bin_counts['schools'].astype(int)
quality_bin_counts['currently_selected'] = quality_bin_counts['currently_selected'].astype(int)
quality_bin_counts['not_currently_selected'] = quality_bin_counts['schools'] - quality_bin_counts['currently_selected']
quality_bin_counts['avg_capacity_score'] = quality_bin_counts['avg_capacity_score'].round(1)
for col in ['recent_ipeds_bachelors', 'recent_bachelor_profiles', 'recent_cip4_profiles', 'current_cip4_profiles']:
    quality_bin_counts[col] = quality_bin_counts[col].round(0).astype(int)

quality_counts_path = OUT_DIR / 'us_school_data_capacity_tier_counts.csv'
quality_bin_counts.to_csv(quality_counts_path, index=False)

# School-level decision file for expansion review. This is intentionally wide
# enough to decide: keep current schools, add high-quality schools, and review
# large/important schools that are below the strict excellent/strong bar.
selection_review = us_school_capacity.copy()
selection_review['size_recent_ipeds_bachelors'] = selection_review['recent_ipeds_bachelors'].round(0).astype(int)
selection_review['size_recent_revelio_bachelor_profiles'] = selection_review['recent_bachelor_profiles'].round(0).astype(int)
selection_review['size_current_student_cip4_profiles'] = selection_review['current_cip4_profiles'].round(0).astype(int)
selection_review['high_quality_flag'] = selection_review['data_capacity_tier'].isin(['excellent', 'strong'])
selection_review['usable_or_better_flag'] = selection_review['data_capacity_tier'].isin(['excellent', 'strong', 'usable'])
selection_review['large_school_flag'] = (
    selection_review['recent_ipeds_bachelors'].fillna(0).ge(5000)
    | selection_review['recent_bachelor_profiles'].fillna(0).ge(5000)
    | selection_review['current_cip4_profiles'].fillna(0).ge(2500)
)
selection_review['add_candidate_flag'] = (
    ~selection_review['currently_selected']
    & (
        selection_review['high_quality_flag']
        | (selection_review['large_school_flag'] & selection_review['usable_or_better_flag'])
    )
)
selection_review['review_candidate_flag'] = (
    ~selection_review['currently_selected']
    & selection_review['large_school_flag']
    & ~selection_review['add_candidate_flag']
)

def _school_expansion_action(row):
    if bool(row['currently_selected']):
        return 'keep_current'
    if bool(row['high_quality_flag']):
        return 'add_high_quality'
    if bool(row['large_school_flag']) and bool(row['usable_or_better_flag']):
        return 'add_large_usable'
    if bool(row['large_school_flag']):
        return 'review_large_lower_quality'
    return 'lower_priority'

selection_review['expansion_action'] = selection_review.apply(_school_expansion_action, axis=1)
selection_review['expansion_sort'] = selection_review['expansion_action'].map({
    'keep_current': 0,
    'add_high_quality': 1,
    'add_large_usable': 2,
    'review_large_lower_quality': 3,
    'lower_priority': 4,
}).fillna(9)

selection_review_cols = [
    'unitid', 'ipeds_name', 'expansion_action', 'currently_selected',
    'add_candidate_flag', 'review_candidate_flag', 'large_school_flag',
    'data_capacity_tier', 'data_capacity_score',
    'size_recent_ipeds_bachelors', 'size_recent_revelio_bachelor_profiles',
    'size_current_student_cip4_profiles',
    'cip4_profile_rate_pct', 'high_conf_cip4_rate_pct',
    'revelio_to_ipeds_profile_rate_pct', 'cip4s_with_10_profiles',
    'one_year_position_rate_pct', 'current_to_recent_profile_rate_pct',
    'ipeds_bachelor_cip4s', 'observed_cip4s', 'current_cip4s',
]
selection_review_path = OUT_DIR / 'us_school_expansion_review.csv'
selection_review = selection_review.sort_values(
    ['expansion_sort', 'data_capacity_score', 'size_recent_ipeds_bachelors', 'size_recent_revelio_bachelor_profiles'],
    ascending=[True, False, False, False],
).reset_index(drop=True)
selection_review[selection_review_cols].to_csv(selection_review_path, index=False)

print('\nU.S. bachelor-granting school data-capacity tiers:')
print(quality_bin_counts[['data_capacity_tier', 'schools', 'currently_selected', 'not_currently_selected', 'avg_capacity_score']].to_string(index=False))
print(f"\nSchools usable or better: {us_school_capacity['data_capacity_tier'].isin(['excellent', 'strong', 'usable']).sum():,}")
print(f"Schools strong or excellent: {us_school_capacity['data_capacity_tier'].isin(['excellent', 'strong']).sum():,}")
print(f"Currently selected schools: {us_school_capacity['currently_selected'].sum():,}")
print(f"Strong/excellent schools not currently selected: {((us_school_capacity['data_capacity_tier'].isin(['excellent', 'strong'])) & ~us_school_capacity['currently_selected']).sum():,}")
print(f'Wrote {capacity_path}')
print(f'Wrote {quality_counts_path}')
print(f'Wrote {selection_review_path}')

print('\nTop strong/excellent schools not currently selected:')
display_cols = [
    'unitid', 'ipeds_name', 'data_capacity_tier', 'data_capacity_score',
    'recent_cip4_profiles', 'cip4s_with_10_profiles', 'one_year_position_rate_pct',
    'current_cip4_profiles'
]
display(us_school_capacity[
    us_school_capacity['data_capacity_tier'].isin(['excellent', 'strong'])
    & ~us_school_capacity['currently_selected']
][display_cols].head(75))


# Promote the desired school list into the actual precompute school list when requested.
# Prefer the explicit 1,400-school run list in ./csvs; otherwise fall back to the
# older excellent/strong capacity selection. Downstream fact cells use UNITID_LIST
# and UNITID_SQL, so this is the canonical switch for the run.
USE_US_STRONG_EXCELLENT_SCHOOLS = os.environ.get('OUTCOMES_USE_US_STRONG_EXCELLENT_SCHOOLS', '1').strip().lower() not in {'0', 'false', 'no'}
US_CAPACITY_SELECTION_TIERS = {'excellent', 'strong'}
EXPANDED_RUN_LIST_PATHS = [
    DATA_DIR / 'recommended_1400_school_run_list.csv',
    Path('recommended_1400_school_run_list.csv'),
]
expanded_run_list_path = next((path for path in EXPANDED_RUN_LIST_PATHS if path.exists()), None)

if USE_US_STRONG_EXCELLENT_SCHOOLS and expanded_run_list_path is not None:
    explicit = pd.read_csv(expanded_run_list_path, dtype=str)
    explicit.columns = [c.lower().strip() for c in explicit.columns]
    if 'unitid' not in explicit.columns:
        raise ValueError(f'{expanded_run_list_path} has no unitid column')
    explicit['unitid'] = explicit['unitid'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    explicit = explicit[explicit['unitid'].ne('')].drop_duplicates('unitid', keep='first').copy()
    if 'selection_rank' in explicit.columns:
        explicit['selection_rank'] = pd.to_numeric(explicit['selection_rank'], errors='coerce')
        explicit = explicit.sort_values(['selection_rank', 'unitid'], na_position='last').copy()
    else:
        explicit['selection_rank'] = range(1, len(explicit) + 1)
    if 'ipeds_name' not in explicit.columns:
        explicit['ipeds_name'] = explicit['unitid'].map(dict(zip(us_school_capacity['unitid'], us_school_capacity['ipeds_name'])))
    if 'selection_tier' not in explicit.columns:
        source_col = 'selection_source_1400' if 'selection_source_1400' in explicit.columns else 'data_capacity_tier'
        explicit['selection_tier'] = explicit.get(source_col, 'expanded_1400').fillna('expanded_1400').astype(str)
    if 'selection_source' not in explicit.columns:
        explicit['selection_source'] = 'explicit_1400_school_run_list'
    for col in [
        'data_capacity_tier', 'data_capacity_score', 'recent_ipeds_bachelors',
        'recent_bachelor_profiles', 'recent_cip4_profiles', 'cip4_profile_rate_pct',
        'one_year_position_rate_pct', 'current_cip4_profiles'
    ]:
        if col not in explicit.columns:
            explicit[col] = np.nan

    school_meta = explicit[[
        'selection_rank', 'unitid', 'ipeds_name', 'selection_tier', 'selection_source',
        'data_capacity_tier', 'data_capacity_score', 'recent_ipeds_bachelors',
        'recent_bachelor_profiles', 'recent_cip4_profiles', 'cip4_profile_rate_pct',
        'one_year_position_rate_pct', 'current_cip4_profiles'
    ]].copy()
    school_meta['selection_rank'] = range(1, len(school_meta) + 1)
    UNITID_LIST = school_meta['unitid'].astype(str).tolist()
    UNITID_SQL = ','.join(f"'{u}'" for u in UNITID_LIST)
    selection_source = f'explicit 1,400-school run list: {expanded_run_list_path}'

    selected_capacity_path = OUT_DIR / 'us_1400_school_run_list.csv'
    school_meta.to_csv(selected_capacity_path, index=False)
    print(f"\nUsing explicit 1,400-school run list for precompute: {len(UNITID_LIST):,} schools.")
    print(f"Selection source: {selection_source}")
    print(f"Wrote {selected_capacity_path}")
elif USE_US_STRONG_EXCELLENT_SCHOOLS:
    selected_capacity = us_school_capacity[
        us_school_capacity['data_capacity_tier'].isin(US_CAPACITY_SELECTION_TIERS)
    ].copy()
    selected_capacity = selected_capacity.sort_values(
        ['data_capacity_tier', 'data_capacity_score', 'recent_cip4_profiles'],
        ascending=[True, False, False],
    ).reset_index(drop=True)
    selected_capacity['selection_rank'] = range(1, len(selected_capacity) + 1)
    selected_capacity['selection_tier'] = 'capacity_' + selected_capacity['data_capacity_tier'].astype(str)
    selected_capacity['selection_source'] = 'us_school_data_capacity_audit'

    school_meta = selected_capacity[[
        'selection_rank', 'unitid', 'ipeds_name', 'selection_tier', 'selection_source',
        'data_capacity_tier', 'data_capacity_score', 'recent_ipeds_bachelors',
        'recent_bachelor_profiles', 'recent_cip4_profiles', 'cip4_profile_rate_pct',
        'one_year_position_rate_pct', 'current_cip4_profiles'
    ]].copy()
    UNITID_LIST = school_meta['unitid'].astype(str).tolist()
    UNITID_SQL = ','.join(f"'{u}'" for u in UNITID_LIST)
    selection_source = 'U.S. data-capacity audit: excellent + strong schools'

    selected_capacity_path = OUT_DIR / 'us_strong_excellent_school_run_list.csv'
    school_meta.to_csv(selected_capacity_path, index=False)
    print(f"\nUsing all strong/excellent U.S. schools for precompute: {len(UNITID_LIST):,} schools.")
    print(f"Selection source: {selection_source}")
    print(f"Wrote {selected_capacity_path}")
else:
    print('\nKeeping the earlier NACE/elite UNITID_LIST; capacity audit remains diagnostic only.')


In [3]:
# =============================================================
# CELL 2: Build calibrated graduate spine + base education/position join
# =============================================================
# IPEDS is used here only as a calibration signal. It is not meant to be
# displayed as a front-end coverage product. Position weights are applied
# at the selected position row, then multiplied by the IPEDS major-mix
# calibration factor for final outcome weights.

conn = sfClient.connect()
cur = conn.cursor()

# --- Upload CIP title reference tables to Snowflake ---
print('Uploading CIP title reference tables...')

def upload_cip_titles(cur, df, table_name, code_col, title_col):
    cur.execute(f"CREATE OR REPLACE TABLE {table_name} (code VARCHAR, title VARCHAR)")
    rows = [(str(r[code_col]), str(r[title_col])) for _, r in df.iterrows()]
    batch_size = 5000
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        cur.executemany(f"INSERT INTO {table_name} (code, title) VALUES (%s, %s)", batch)
    print(f'  {table_name}: {len(rows):,} rows')

upload_cip_titles(cur, cip2_df, f'{SCRATCH}.CIP2_TITLES', 'cip2', 'cip2_title')
upload_cip_titles(cur, cip4_df, f'{SCRATCH}.CIP4_TITLES', 'cip4', 'cip4_title')
upload_cip_titles(cur, cip6_df, f'{SCRATCH}.CIP6_TITLES', 'cip6', 'cip6_title')
print('Done uploading CIP title tables.')

# --- Upload a compact IPEDS CIP4 denominator for the selected schools ---
degree_map_calibration = pd.DataFrame([
    ('03', 'Associates'),
    ('05', 'Bachelors'),
    ('07', 'Masters'),
    ('17', 'Research Doctorate'),
    ('18', 'Professional Doctorate'),
    ('09', 'Professional Doctorate'),
    ('9', 'Professional Doctorate'),
    ('10', 'Professional Doctorate'),
    ('11', 'Professional Doctorate'),
    ('19', 'Other Doctorate'),
], columns=['awlevel', 'ipeds_degree_level'])

ipeds_calibration_input = comp.copy()
ipeds_calibration_input['unitid'] = ipeds_calibration_input['unitid'].astype(str)
ipeds_calibration_input['awlevel'] = ipeds_calibration_input['awlevel'].astype(str)
ipeds_calibration_input['year'] = pd.to_numeric(ipeds_calibration_input['year'], errors='coerce')
ipeds_calibration_input['ctotalt'] = pd.to_numeric(ipeds_calibration_input['ctotalt'], errors='coerce').fillna(0)
ipeds_calibration_input['cipcode'] = ipeds_calibration_input['cipcode'].astype(str).str.replace(r'^="?|"$', '', regex=True)
ipeds_calibration_input = ipeds_calibration_input[
    ipeds_calibration_input['unitid'].isin(set(UNITID_LIST))
    & ipeds_calibration_input['year'].between(2005, 2025)
    & ipeds_calibration_input['cipcode'].str.match(r'^\d{2}\.\d{2}', na=False)
    & ipeds_calibration_input['ctotalt'].gt(0)
].copy()
ipeds_calibration_input['cip2'] = ipeds_calibration_input['cipcode'].str[:2]
ipeds_calibration_input['cip4'] = ipeds_calibration_input['cipcode'].str[:5]
ipeds_calibration_input = ipeds_calibration_input[ipeds_calibration_input['cip2'] != '99'].copy()
ipeds_calibration_input = ipeds_calibration_input.merge(degree_map_calibration, on='awlevel', how='inner')
ipeds_calibration_input['cohort_year'] = ipeds_calibration_input['year'].astype(int)
ipeds_calibration_input['cohort_band'] = np.select(
    [
        ipeds_calibration_input['cohort_year'].between(2005, 2009),
        ipeds_calibration_input['cohort_year'].between(2010, 2014),
        ipeds_calibration_input['cohort_year'].between(2015, 2019),
        ipeds_calibration_input['cohort_year'].between(2020, 2025),
    ],
    ['2005-2009', '2010-2014', '2015-2019', '2020-2025'],
    default=None,
)
ipeds_calibration_input = (
    ipeds_calibration_input
    .groupby(['unitid', 'ipeds_degree_level', 'cohort_year', 'cohort_band', 'cip2', 'cip4'], as_index=False)['ctotalt']
    .sum()
    .rename(columns={'ctotalt': 'ipeds_completions'})
)

IPEDS_CIP4_TABLE = f'{SCRATCH}.SCHOOL_IPEDS_CIP4_COMPLETIONS'
cur.execute(f"""
    CREATE OR REPLACE TABLE {IPEDS_CIP4_TABLE} (
        unitid VARCHAR,
        ipeds_degree_level VARCHAR,
        cohort_year INTEGER,
        cohort_band VARCHAR,
        cip2 VARCHAR,
        cip4 VARCHAR,
        ipeds_completions FLOAT
    )
""")
rows = list(ipeds_calibration_input[['unitid', 'ipeds_degree_level', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'ipeds_completions']].itertuples(index=False, name=None))
for i in range(0, len(rows), 5000):
    cur.executemany(
        f"INSERT INTO {IPEDS_CIP4_TABLE} (unitid, ipeds_degree_level, cohort_year, cohort_band, cip2, cip4, ipeds_completions) VALUES (%s, %s, %s, %s, %s, %s, %s)",
        rows[i:i+5000],
    )
print(f'Uploaded IPEDS calibration input: {len(rows):,} school-degree-year-CIP4 rows')

print('Building calibrated graduate spine...')
cur.execute(f"""
CREATE OR REPLACE TABLE {SCRATCH}.SCHOOL_GRADS_CALIBRATED AS
WITH grads AS (
    SELECT
        e.user_id,
        e.unitid,
        e.ipeds_name,
        {assigned_cip4_sql('e')} AS cip_code,
        LEFT({assigned_cip4_sql('e')}, 2) AS cip2,
        {assigned_cip4_sql('e')} AS cip4,
        {candidate_cip6_sql('e')} AS cip6,
        COALESCE(c2.title, '') AS cip2_title,
        COALESCE(c4.title, {assigned_cip_title_sql('e')}, '') AS cip4_title,
        COALESCE(c6.title, '') AS cip6_title,
        COALESCE(c4.title, {assigned_cip_title_sql('e')}, '') AS cip_title,
        e.cip_probability,
        e.match_source,
        e.cip_match_type,
        CASE WHEN e.cip_probability >= 0.8 THEN 1 ELSE 0 END AS high_conf_major_flag,
        {degree_label_sql('e')} AS degree,
        e.enddate AS grad_date,
        YEAR(e.enddate) AS cohort_year,
        CASE
            WHEN YEAR(e.enddate) BETWEEN 2005 AND 2009 THEN '2005-2009'
            WHEN YEAR(e.enddate) BETWEEN 2010 AND 2014 THEN '2010-2014'
            WHEN YEAR(e.enddate) BETWEEN 2015 AND 2019 THEN '2015-2019'
            WHEN YEAR(e.enddate) BETWEEN 2020 AND 2025 THEN '2020-2025'
        END AS cohort_band
    FROM {EDUCATION_CIP} e
    LEFT JOIN {SCRATCH}.CIP2_TITLES c2
        ON LEFT({assigned_cip4_sql('e')}, 2) = c2.code
    LEFT JOIN {SCRATCH}.CIP4_TITLES c4
        ON {assigned_cip4_sql('e')} = c4.code
    LEFT JOIN {SCRATCH}.CIP6_TITLES c6
        ON {candidate_cip6_sql('e')} = c6.code
    WHERE e.unitid IN ({UNITID_SQL})
      AND (e.degree IN ('Associate', 'Bachelor', 'Master', 'MBA') OR e.degree = 'Doctor')
      AND e.enddate IS NOT NULL
      AND YEAR(e.enddate) BETWEEN 2005 AND 2025
),
observed_exact AS (
    SELECT CAST(unitid AS VARCHAR) AS unitid, degree, cohort_year, cohort_band, cip2, cip4,
           COUNT(DISTINCT user_id) AS observed_completions
    FROM grads
    WHERE cip4 IS NOT NULL
    GROUP BY CAST(unitid AS VARCHAR), degree, cohort_year, cohort_band, cip2, cip4
),
calibration_exact AS (
    SELECT
        o.*,
        i.ipeds_completions,
        CASE
            WHEN o.observed_completions >= 10 AND i.ipeds_completions > 0 THEN
                1 + LEAST(1.0, o.observed_completions / 50.0) *
                (LEAST(4.0, GREATEST(0.25, i.ipeds_completions / NULLIF(o.observed_completions, 0))) - 1)
        END AS factor
    FROM observed_exact o
    LEFT JOIN {IPEDS_CIP4_TABLE} i
      ON o.unitid = i.unitid
     AND o.degree = i.ipeds_degree_level
     AND o.cohort_year = i.cohort_year
     AND o.cip4 = i.cip4
),
observed_school_band AS (
    SELECT unitid, degree, cohort_band, cip2, cip4, SUM(observed_completions) AS observed_completions
    FROM observed_exact
    GROUP BY unitid, degree, cohort_band, cip2, cip4
),
ipeds_school_band AS (
    SELECT unitid, ipeds_degree_level AS degree, cohort_band, cip2, cip4, SUM(ipeds_completions) AS ipeds_completions
    FROM {IPEDS_CIP4_TABLE}
    GROUP BY unitid, ipeds_degree_level, cohort_band, cip2, cip4
),
calibration_school_band AS (
    SELECT
        o.*,
        CASE
            WHEN o.observed_completions >= 25 AND i.ipeds_completions > 0 THEN
                1 + LEAST(1.0, o.observed_completions / 100.0) *
                (LEAST(4.0, GREATEST(0.25, i.ipeds_completions / NULLIF(o.observed_completions, 0))) - 1)
        END AS factor
    FROM observed_school_band o
    LEFT JOIN ipeds_school_band i
      ON o.unitid = i.unitid
     AND o.degree = i.degree
     AND o.cohort_band = i.cohort_band
     AND o.cip4 = i.cip4
),
observed_global_year AS (
    SELECT degree, cohort_year, cip2, cip4, SUM(observed_completions) AS observed_completions
    FROM observed_exact
    GROUP BY degree, cohort_year, cip2, cip4
),
ipeds_global_year AS (
    SELECT ipeds_degree_level AS degree, cohort_year, cip2, cip4, SUM(ipeds_completions) AS ipeds_completions
    FROM {IPEDS_CIP4_TABLE}
    GROUP BY ipeds_degree_level, cohort_year, cip2, cip4
),
calibration_global_year AS (
    SELECT
        o.*,
        CASE
            WHEN o.observed_completions >= 50 AND i.ipeds_completions > 0 THEN
                1 + LEAST(1.0, o.observed_completions / 150.0) *
                (LEAST(4.0, GREATEST(0.25, i.ipeds_completions / NULLIF(o.observed_completions, 0))) - 1)
        END AS factor
    FROM observed_global_year o
    LEFT JOIN ipeds_global_year i
      ON o.degree = i.degree
     AND o.cohort_year = i.cohort_year
     AND o.cip4 = i.cip4
),
observed_global_band AS (
    SELECT degree, cohort_band, cip2, cip4, SUM(observed_completions) AS observed_completions
    FROM observed_exact
    GROUP BY degree, cohort_band, cip2, cip4
),
ipeds_global_band AS (
    SELECT ipeds_degree_level AS degree, cohort_band, cip2, cip4, SUM(ipeds_completions) AS ipeds_completions
    FROM {IPEDS_CIP4_TABLE}
    GROUP BY ipeds_degree_level, cohort_band, cip2, cip4
),
calibration_global_band AS (
    SELECT
        o.*,
        CASE
            WHEN o.observed_completions >= 75 AND i.ipeds_completions > 0 THEN
                1 + LEAST(1.0, o.observed_completions / 200.0) *
                (LEAST(4.0, GREATEST(0.25, i.ipeds_completions / NULLIF(o.observed_completions, 0))) - 1)
        END AS factor
    FROM observed_global_band o
    LEFT JOIN ipeds_global_band i
      ON o.degree = i.degree
     AND o.cohort_band = i.cohort_band
     AND o.cip4 = i.cip4
)
SELECT
    g.*,
    COALESCE(ce.factor, csb.factor, cgy.factor, cgb.factor, 1.0) AS ipeds_calibration_weight,
    CASE
        WHEN ce.factor IS NOT NULL THEN 'school_year_cip4'
        WHEN csb.factor IS NOT NULL THEN 'school_band_cip4'
        WHEN cgy.factor IS NOT NULL THEN 'global_year_cip4'
        WHEN cgb.factor IS NOT NULL THEN 'global_band_cip4'
        ELSE 'none'
    END AS ipeds_calibration_source,
    ce.observed_completions AS calibration_observed_completions,
    ce.ipeds_completions AS calibration_ipeds_completions
FROM grads g
LEFT JOIN calibration_exact ce
  ON CAST(g.unitid AS VARCHAR) = ce.unitid
 AND g.degree = ce.degree
 AND g.cohort_year = ce.cohort_year
 AND g.cip4 = ce.cip4
LEFT JOIN calibration_school_band csb
  ON CAST(g.unitid AS VARCHAR) = csb.unitid
 AND g.degree = csb.degree
 AND g.cohort_band = csb.cohort_band
 AND g.cip4 = csb.cip4
LEFT JOIN calibration_global_year cgy
  ON g.degree = cgy.degree
 AND g.cohort_year = cgy.cohort_year
 AND g.cip4 = cgy.cip4
LEFT JOIN calibration_global_band cgb
  ON g.degree = cgb.degree
 AND g.cohort_band = cgb.cohort_band
 AND g.cip4 = cgb.cip4
""")

print('Building base outcomes table in Snowflake...')
print('This joins calibrated education records to positions at 1/5/10 year horizons.')
print(f'Salaries will be inflation-adjusted to constant {SALARY_REAL_BASE_YEAR} dollars.')
print(f'Position weight column: {POSITION_WEIGHT_COL or "none found; using 1.0"}')
print('May take several minutes.\n')

cur.execute(f"""
CREATE OR REPLACE TABLE {SCRATCH}.SCHOOL_OUTCOMES_BASE AS
WITH horizons AS (
    SELECT 1 AS horizon UNION ALL SELECT 5 UNION ALL SELECT 10
),
grad_horizons AS (
    SELECT
        g.*,
        h.horizon,
        DATEADD('year', h.horizon, g.grad_date) AS target_date
    FROM {SCRATCH}.SCHOOL_GRADS_CALIBRATED g
    CROSS JOIN horizons h
    WHERE DATEADD('year', h.horizon, g.grad_date) <= CURRENT_DATE()
),
cpi_u AS (
    SELECT column1 AS cpi_year, column2 AS cpi_u_avg
    FROM VALUES
        (2005, 195.300), (2006, 201.600), (2007, 207.342), (2008, 215.303),
        (2009, 214.537), (2010, 218.056), (2011, 224.939), (2012, 229.594),
        (2013, 232.957), (2014, 236.736), (2015, 237.017), (2016, 240.007),
        (2017, 245.120), (2018, 251.107), (2019, 255.657), (2020, 258.811),
        (2021, 270.970), (2022, 292.655), (2023, 304.702), (2024, 313.689),
        (2025, 321.943)
),
position_match AS (
    SELECT
        gh.*,
        p.salary AS salary_nominal,
        CASE
            WHEN p.salary IS NOT NULL AND cpi.cpi_u_avg IS NOT NULL
            THEN ROUND(p.salary * {SALARY_REAL_BASE_CPI} / cpi.cpi_u_avg, 0)
            ELSE NULL
        END AS salary,
        p.company_name,
        p.ultimate_parent_company_name,
        p.metro_area,
        p.msa,
        p.city,
        p.state,
        p.country,
        p.role_k10_v3,
        p.role_k50_v3,
        p.role_k150_v3,
        p.role_k500_v3,
        p.rics_k50,
        p.rics_k200,
        p.rics_k400,
        p.naics_code,
        p.naics_description,
        p.seniority,
        p.is_primary,
        p.title_raw,
        GREATEST(0.0, {position_weight_sql('p')}) AS position_weight,
        COALESCE(gh.ipeds_calibration_weight, 1.0) AS education_weight,
        GREATEST(0.0, {position_weight_sql('p')}) * COALESCE(gh.ipeds_calibration_weight, 1.0) AS analysis_weight,
        ROW_NUMBER() OVER (
            PARTITION BY gh.user_id, gh.unitid, gh.degree, gh.cohort_year, gh.horizon
            ORDER BY
                p.is_primary DESC NULLS LAST,
                p.salary DESC NULLS LAST,
                p.startdate DESC
        ) AS pos_rank
    FROM grad_horizons gh
    JOIN {POSITION_TABLE} p
        ON gh.user_id = p.user_id
        AND p.startdate <= gh.target_date
        AND (p.enddate >= gh.target_date OR p.enddate IS NULL)
    LEFT JOIN cpi_u cpi
        ON YEAR(gh.target_date) = cpi.cpi_year
)
SELECT *
FROM position_match
WHERE pos_rank = 1
""")

print('Done! Checking row counts and calibration mix...')

counts = sfClient.load_df(f"""
    SELECT horizon,
           COUNT(*) AS raw_n,
           ROUND(SUM(analysis_weight), 0) AS weighted_n,
           COUNT(salary) AS raw_salary_obs,
           ROUND(SUM(CASE WHEN salary IS NOT NULL THEN analysis_weight ELSE 0 END), 0) AS weighted_salary_obs,
           ROUND(SUM(salary * analysis_weight) / NULLIF(SUM(CASE WHEN salary IS NOT NULL THEN analysis_weight ELSE 0 END), 0), 0) AS weighted_avg_salary_real,
           COUNT(DISTINCT user_id) AS unique_users,
           COUNT(DISTINCT unitid) AS schools
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
    GROUP BY horizon ORDER BY horizon
""")
print(counts.to_string(index=False))

calibration_counts = sfClient.load_df(f"""
    SELECT ipeds_calibration_source, COUNT(*) AS raw_grads,
           ROUND(AVG(ipeds_calibration_weight), 3) AS avg_factor,
           ROUND(MIN(ipeds_calibration_weight), 3) AS min_factor,
           ROUND(MAX(ipeds_calibration_weight), 3) AS max_factor
    FROM {SCRATCH}.SCHOOL_GRADS_CALIBRATED
    GROUP BY ipeds_calibration_source
    ORDER BY raw_grads DESC
""")
print('\nCalibration source mix:')
print(calibration_counts.to_string(index=False))

cur.close()
conn.close()


Uploading CIP title reference tables...
  USER_CAELAN.TMP_MONTHLY.CIP2_TITLES: 50 rows
  USER_CAELAN.TMP_MONTHLY.CIP4_TITLES: 473 rows
  USER_CAELAN.TMP_MONTHLY.CIP6_TITLES: 2,325 rows
Done uploading CIP title tables.
Building base outcomes table in Snowflake...
This joins education records to positions at 1/5/10 year horizons.
Salaries will be inflation-adjusted to constant 2024 dollars.
May take several minutes.



Done! Checking row counts...
 horizon      n  has_salary  avg_salary_real  unique_users  schools
       1 793155      744966         107807.0        732279        7
       5 697829      648206         127997.0        640734        7
      10 480975      435566         151443.0        445518        7

Cohort coverage:
cohort_band  horizon      n
  2005-2009        1 115696
  2005-2009        5 150989
  2005-2009       10 166197
  2010-2014        1 172892
  2010-2014        5 210336
  2010-2014       10 223748
  2015-2019        1 207492
  2015-2019        5 235397
  2015-2019       10  91030
  2020-2025        1 297075
  2020-2025        5 101107


In [4]:
# =============================================================
# CELL 3: weighted school_earnings_fact + annual earnings curves + overview
# =============================================================
# All counts, shares, averages, and salary quantiles are weighted by:
# position_weight x IPEDS-derived education calibration factor.

earnings_parts = []

EARNINGS_QUANTILES = {0.10: 'p10', 0.25: 'p25', 0.50: 'median', 0.75: 'p75', 0.90: 'p90'}
EARNINGS_HAVING = "SUM(CASE WHEN salary IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END) >= 5"

CIP_SOURCES = {
    'ALL': """
        SELECT
            user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
            'ALL' AS cip_level, 'ALL' AS cip_code, 'All Majors' AS cip_title,
            salary, seniority, analysis_weight
        FROM {scratch}.SCHOOL_OUTCOMES_BASE
    """,
    'CIP2': """
        SELECT
            user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
            'CIP2' AS cip_level, COALESCE(cip2, 'XX') AS cip_code, cip2_title AS cip_title,
            salary, seniority, analysis_weight
        FROM {scratch}.SCHOOL_OUTCOMES_BASE
    """,
    'CIP4': """
        SELECT
            user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
            'CIP4' AS cip_level, cip4 AS cip_code, cip4_title AS cip_title,
            salary, seniority, analysis_weight
        FROM {scratch}.SCHOOL_OUTCOMES_BASE
        WHERE cip4 IS NOT NULL
    """,
    'CIP6_TOP': """
        SELECT
            user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
            'CIP6_TOP' AS cip_level, cip6 AS cip_code, cip6_title AS cip_title,
            salary, seniority, analysis_weight
        FROM {scratch}.SCHOOL_OUTCOMES_BASE
        WHERE cip6 IS NOT NULL
    """,
}

earnings_group_cols = ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip_level', 'cip_code', 'cip_title']
for level, source_template in CIP_SOURCES.items():
    print(f'Querying weighted {level}...')
    sql = weighted_aggregate_sql(
        source_template.format(scratch=SCRATCH),
        earnings_group_cols,
        count_alias='n_alumni',
        salary_count_alias='n_salary',
        quantile_aliases=EARNINGS_QUANTILES,
        mean_alias='mean',
        having=EARNINGS_HAVING,
    )
    df = sfClient.load_df(sql)
    df.columns = [c.lower() for c in df.columns]
    earnings_parts.append(df)
    print(f'  {level}: {len(df):,} rows')

earnings = pd.concat(earnings_parts, ignore_index=True)

print(f'\nTotal school_earnings_fact: {len(earnings):,} rows')
print(f'  By level: {earnings.groupby("cip_level").size().to_dict()}')

sample = earnings[(earnings['ipeds_name']=='University of California-Berkeley') &
                  (earnings['degree']=='Bachelors') &
                  (earnings['horizon']==5) &
                  (earnings['cip_level']=='ALL')].sort_values('cohort_year')
print('\nSample — Berkeley, Bachelors, 5yr, ALL:')
print(sample[['cohort_year','n_alumni','raw_n','n_salary','median','p75','p90']].to_string(index=False))

write_fact_parquet(earnings, OUT_DIR / 'school_earnings_fact.parquet')
print('\nSaved school_earnings_fact.parquet.')

# Annual earnings curves for cohort-vs-cohort comparisons.
print('\nBuilding weighted annual earnings curve...')
annual_base_source = f"""
    WITH annual_horizons AS (
        SELECT column1 AS horizon
        FROM VALUES (0), (1), (2), (3), (4), (5), (6), (7), (8), (9), (10)
    ),
    grad_horizons AS (
        SELECT
            g.*,
            h.horizon,
            DATEADD('year', h.horizon, g.grad_date) AS target_date
        FROM {SCRATCH}.SCHOOL_GRADS_CALIBRATED g
        CROSS JOIN annual_horizons h
        WHERE DATEADD('year', h.horizon, g.grad_date) <= CURRENT_DATE()
    ),
    cpi_u AS (
        SELECT column1 AS cpi_year, column2 AS cpi_u_avg
        FROM VALUES
            (2005, 195.300), (2006, 201.600), (2007, 207.342), (2008, 215.303),
            (2009, 214.537), (2010, 218.056), (2011, 224.939), (2012, 229.594),
            (2013, 232.957), (2014, 236.736), (2015, 237.017), (2016, 240.007),
            (2017, 245.120), (2018, 251.107), (2019, 255.657), (2020, 258.811),
            (2021, 270.970), (2022, 292.655), (2023, 304.702), (2024, 313.689),
            (2025, 321.943)
    ),
    position_match AS (
        SELECT
            gh.*,
            CASE
                WHEN p.salary IS NOT NULL AND cpi.cpi_u_avg IS NOT NULL
                THEN ROUND(p.salary * {SALARY_REAL_BASE_CPI} / cpi.cpi_u_avg, 0)
                ELSE NULL
            END AS salary,
            p.seniority,
            GREATEST(0.0, {position_weight_sql('p')}) AS position_weight,
            GREATEST(0.0, {position_weight_sql('p')}) * COALESCE(gh.ipeds_calibration_weight, 1.0) AS analysis_weight,
            ROW_NUMBER() OVER (
                PARTITION BY gh.user_id, gh.unitid, gh.degree, gh.cohort_year, gh.horizon
                ORDER BY p.is_primary DESC NULLS LAST, p.salary DESC NULLS LAST, p.startdate DESC
            ) AS pos_rank
        FROM grad_horizons gh
        JOIN {POSITION_TABLE} p
            ON gh.user_id = p.user_id
            AND p.startdate <= gh.target_date
            AND (p.enddate >= gh.target_date OR p.enddate IS NULL)
        LEFT JOIN cpi_u cpi
            ON YEAR(gh.target_date) = cpi.cpi_year
    )
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip4 AS cip_code, cip_title,
        salary, seniority, analysis_weight
    FROM position_match
    WHERE pos_rank = 1
"""
annual_by_major = weighted_aggregate_sql(
    annual_base_source + "\nAND cip4 IS NOT NULL",
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title'],
    count_alias='n',
    salary_count_alias='salary_obs',
    having='SUM(COALESCE(analysis_weight, 1.0)) >= 1',
)
annual_all_major_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        'ALL' AS cip2, 'ALL' AS cip4, 'ALL' AS cip_code, 'All majors' AS cip_title,
        salary, seniority, analysis_weight
    FROM ({annual_base_source}) b
"""
annual_all_major = weighted_aggregate_sql(
    annual_all_major_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title'],
    count_alias='n',
    salary_count_alias='salary_obs',
    having='SUM(COALESCE(analysis_weight, 1.0)) >= 1',
)
annual_earnings_curve = sfClient.load_df(f"""
    SELECT * FROM ({annual_by_major})
    UNION ALL
    SELECT * FROM ({annual_all_major})
""")
annual_earnings_curve.columns = [c.lower() for c in annual_earnings_curve.columns]

print(f'\nTotal school_earnings_curve_fact: {len(annual_earnings_curve):,} rows')
print(f"  Horizon range: {annual_earnings_curve['horizon'].min()} - {annual_earnings_curve['horizon'].max()}")
print(f"  Has class of 2025: {(annual_earnings_curve['cohort_year'] == 2025).any()}")

write_fact_parquet(annual_earnings_curve, OUT_DIR / 'school_earnings_curve_fact.parquet')
print('Saved school_earnings_curve_fact.parquet.')

# School-level overview used by the explorer landing page.
overview_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        salary, seniority, analysis_weight,
        CASE
            WHEN LOWER(COALESCE(ultimate_parent_company_name, '')) = 'government of the united states of america'
                 AND NULLIF(company_name, '') IS NOT NULL
            THEN company_name
            ELSE COALESCE(NULLIF(ultimate_parent_company_name, ''), NULLIF(company_name, ''), 'Unknown')
        END AS employer,
        COALESCE(metro_area, state, country) AS metro_key
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
"""
overview = sfClient.load_df(weighted_aggregate_sql(
    overview_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band'],
    count_alias='observed_alumni',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "COUNT(DISTINCT employer) AS unique_employers",
        "COUNT(DISTINCT metro_key) AS unique_metros",
    ],
))
overview.columns = [c.lower() for c in overview.columns]
print(f'\nTotal school_overview_fact: {len(overview):,} rows')
print('  Degrees:', sorted(overview['degree'].dropna().astype(str).unique().tolist()))
write_fact_parquet(overview, OUT_DIR / 'school_overview_fact.parquet')
print('Saved school_overview_fact.parquet')


Querying ALL...
  ALL: 2,567 rows
Querying CIP2...
  CIP2: 19,137 rows
Querying CIP4...
  CIP4: 31,951 rows
Querying CIP6...
  CIP6: 33,503 rows

Total school_earnings_fact: 87,158 rows
  By level: {'ALL': 2567, 'CIP2': 19137, 'CIP4': 31951, 'CIP6': 33503}

Sample — Berkeley, Bachelors, 5yr, ALL:
 cohort_year  n_alumni  n_salary   median      p75      p90
        2005      3210      3210  97433.0 145674.0 208741.0
        2006      3366      3366  91290.0 141221.0 205249.0
        2007      3700      3700  89705.0 135994.0 194089.0
        2008      4271      4271  90841.0 137116.0 198215.0
        2009      4353      4353  89666.0 133019.0 192554.0
        2010      4750      4750  94627.0 142800.0 201505.0
        2011      4641      4641  96443.0 145468.0 200737.0
        2012      4897      4897  97950.0 144607.0 205041.0
        2013      5124      5124  97076.0 147651.0 204826.0
        2014      5098      5098 101215.0 151547.0 208084.0
        2015      5268      5268 103509.0 


Total school_earnings_curve_fact: 297,087 rows
  Horizon range: 0 - 10
  Has class of 2025: True

Sample curve — Berkeley, Bachelors, ALL:
 cohort_year  horizon  salary_obs  median_salary
        2021        2        6021        93932.0
        2021        3        5916        96449.0
        2021        4        5821        96233.0
        2021        5           0            NaN
        2022        0        5088        76277.0
        2022        1        5759        86964.0
        2022        2        6183        89956.0
        2022        3        6114        91858.0
        2022        4           0            NaN
        2023        0        5416        77769.0
        2023        1        5833        84818.0
        2023        2        6225        86876.0
        2023        3           0            NaN
        2024        0        5302        77623.0
        2024        1        5394        76989.0
        2024        2           0            NaN
        2025        0      

In [5]:
# =============================================================
# CELL 4: weighted school_employer_fact + employer-role drill fact
# =============================================================
# Employer outcomes by school x degree x horizon x cohort x major.
# Counts/shares are weighted; raw_n remains available for diagnostics.

conn = sfClient.connect()
cur = conn.cursor()
cur.execute(f"""
CREATE OR REPLACE TABLE {SCRATCH}.SCHOOL_OUTCOMES_EMPLOYER_ENRICHED AS
WITH base AS (
    SELECT
        user_id,
        unitid,
        ipeds_name,
        degree,
        horizon,
        cohort_year,
        cohort_band,
        cip2,
        cip4,
        cip4 AS cip_code,
        cip_title,
        CASE
            WHEN LOWER(COALESCE(ultimate_parent_company_name, '')) = 'government of the united states of america'
                 AND NULLIF(company_name, '') IS NOT NULL
            THEN company_name
            ELSE COALESCE(NULLIF(ultimate_parent_company_name, ''), NULLIF(company_name, ''), 'Unknown')
        END AS employer,
        CASE
            WHEN COALESCE(NULLIF(ultimate_parent_company_name, ''), NULLIF(company_name, '')) IS NULL THEN 1 ELSE 0
        END AS unknown_employer_flag,
        CASE
            WHEN COALESCE(NULLIF(ultimate_parent_company_name, ''), NULLIF(company_name, '')) IS NULL THEN 0 ELSE 1
        END AS named_employer_flag,
        role_k10_v3,
        role_k50_v3,
        role_k150_v3,
        rics_k50 AS industry_k50,
        rics_k200 AS industry_k200,
        salary,
        seniority,
        analysis_weight,
        LOWER(REGEXP_REPLACE(ipeds_name, '[^a-z0-9]', '')) AS school_norm,
        LOWER(REGEXP_REPLACE(
            CASE
                WHEN LOWER(COALESCE(ultimate_parent_company_name, '')) = 'government of the united states of america'
                     AND NULLIF(company_name, '') IS NOT NULL
                THEN company_name
                ELSE COALESCE(NULLIF(ultimate_parent_company_name, ''), NULLIF(company_name, ''), 'Unknown')
            END,
            '[^a-z0-9]',
            ''
        )) AS employer_norm
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
),
enriched AS (
    SELECT
        *,
        CASE
            WHEN named_employer_flag = 1
             AND employer_norm <> ''
             AND (
                employer_norm = school_norm
                OR employer_norm LIKE '%' || school_norm || '%'
                OR school_norm LIKE '%' || employer_norm || '%'
                OR (CAST(unitid AS VARCHAR) = '190150' AND employer_norm = 'thetrusteesofcolumbiauniversityinthecityofnewyork')
             )
            THEN 1 ELSE 0
        END AS same_school_employer_flag
    FROM base
)
SELECT
    *,
    CASE WHEN named_employer_flag = 1 AND same_school_employer_flag = 0 THEN 1 ELSE 0 END AS career_employer_flag
FROM enriched
""")
cur.close()
conn.close()

employer_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip_code, cip_title, employer,
        salary, seniority, analysis_weight,
        unknown_employer_flag, named_employer_flag, same_school_employer_flag, career_employer_flag
    FROM {SCRATCH}.SCHOOL_OUTCOMES_EMPLOYER_ENRICHED
"""
employer_by_major = weighted_aggregate_sql(
    employer_source + "\nWHERE cip_code IS NOT NULL",
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'employer'],
    count_alias='n',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "ROUND(SUM(unknown_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS unknown_employer_n",
        "ROUND(SUM(named_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS named_employer_n",
        "ROUND(SUM(same_school_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS same_school_employer_n",
        "ROUND(SUM(career_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS career_employer_n",
    ],
)
employer_all_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        'ALL' AS cip2, 'ALL' AS cip4, 'ALL' AS cip_code, 'All majors' AS cip_title, employer,
        salary, seniority, analysis_weight,
        unknown_employer_flag, named_employer_flag, same_school_employer_flag, career_employer_flag
    FROM {SCRATCH}.SCHOOL_OUTCOMES_EMPLOYER_ENRICHED
"""
employer_all_major = weighted_aggregate_sql(
    employer_all_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'employer'],
    count_alias='n',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "ROUND(SUM(unknown_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS unknown_employer_n",
        "ROUND(SUM(named_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS named_employer_n",
        "ROUND(SUM(same_school_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS same_school_employer_n",
        "ROUND(SUM(career_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS career_employer_n",
    ],
)
employer = sfClient.load_df(f"""
    WITH stacked AS (
        SELECT * FROM ({employer_by_major})
        UNION ALL
        SELECT * FROM ({employer_all_major})
    ),
    totals AS (
        SELECT
            unitid, degree, horizon, cohort_year, cip2, cip4, cip_code,
            SUM(weighted_n) AS total_positions,
            SUM(named_employer_n) AS total_named_positions,
            SUM(same_school_employer_n) AS total_same_school_positions,
            SUM(career_employer_n) AS total_career_positions
        FROM stacked
        GROUP BY unitid, degree, horizon, cohort_year, cip2, cip4, cip_code
    ),
    ranked AS (
        SELECT
            s.*,
            t.total_positions,
            t.total_named_positions,
            t.total_same_school_positions,
            t.total_career_positions,
            ROUND(s.weighted_n / NULLIF(t.total_positions, 0) * 100, 2) AS share_pct,
            ROUND(s.weighted_n / NULLIF(t.total_named_positions, 0) * 100, 2) AS share_pct_named,
            ROUND(s.weighted_n / NULLIF(t.total_career_positions, 0) * 100, 2) AS share_pct_career,
            ROW_NUMBER() OVER (
                PARTITION BY s.unitid, s.degree, s.horizon, s.cohort_year, s.cip2, s.cip4, s.cip_code
                ORDER BY s.weighted_n DESC, s.employer
            ) AS employer_rank
        FROM stacked s
        JOIN totals t
          ON s.unitid = t.unitid
         AND s.degree = t.degree
         AND s.horizon = t.horizon
         AND s.cohort_year = t.cohort_year
         AND s.cip2 = t.cip2
         AND s.cip4 = t.cip4
         AND s.cip_code = t.cip_code
    )
    SELECT * FROM ranked
""")
employer.columns = [c.lower() for c in employer.columns]

print(f'school_employer_fact: {len(employer):,} rows')
print('\nTop employers at Berkeley (Bachelors, 5yr, 2018, all majors):')
h_emp = employer[(employer['ipeds_name']=='University of California-Berkeley') &
                 (employer['degree']=='Bachelors') &
                 (employer['horizon']==5) &
                 (employer['cohort_year']==2018) &
                 (employer['cip_code']=='ALL')].head(10)
print(h_emp[['employer', 'n', 'raw_n', 'share_pct', 'share_pct_career', 'same_school_employer_n', 'unknown_employer_n']].to_string(index=False))

write_fact_parquet(employer, OUT_DIR / 'school_employer_fact.parquet')
print('\nSaved school_employer_fact.parquet')

role_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip_code, cip_title, employer,
        role_k10_v3, role_k50_v3, role_k150_v3, industry_k50, industry_k200,
        salary, seniority, analysis_weight,
        unknown_employer_flag, named_employer_flag, same_school_employer_flag, career_employer_flag
    FROM {SCRATCH}.SCHOOL_OUTCOMES_EMPLOYER_ENRICHED
"""
role_by_major = weighted_aggregate_sql(
    role_source + "\nWHERE cip_code IS NOT NULL",
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'employer', 'role_k10_v3', 'role_k50_v3', 'role_k150_v3', 'industry_k50', 'industry_k200'],
    count_alias='n',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "ROUND(SUM(unknown_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS unknown_employer_n",
        "ROUND(SUM(named_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS named_employer_n",
        "ROUND(SUM(same_school_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS same_school_employer_n",
        "ROUND(SUM(career_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS career_employer_n",
    ],
)
role_all_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        'ALL' AS cip2, 'ALL' AS cip4, 'ALL' AS cip_code, 'All majors' AS cip_title, employer,
        role_k10_v3, role_k50_v3, role_k150_v3, industry_k50, industry_k200,
        salary, seniority, analysis_weight,
        unknown_employer_flag, named_employer_flag, same_school_employer_flag, career_employer_flag
    FROM {SCRATCH}.SCHOOL_OUTCOMES_EMPLOYER_ENRICHED
"""
role_all_major = weighted_aggregate_sql(
    role_all_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'employer', 'role_k10_v3', 'role_k50_v3', 'role_k150_v3', 'industry_k50', 'industry_k200'],
    count_alias='n',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "ROUND(SUM(unknown_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS unknown_employer_n",
        "ROUND(SUM(named_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS named_employer_n",
        "ROUND(SUM(same_school_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS same_school_employer_n",
        "ROUND(SUM(career_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS career_employer_n",
    ],
)
employer_role = sfClient.load_df(f"""
    WITH employer_rows AS (
        SELECT
            s.unitid,
            s.degree,
            s.horizon,
            s.cohort_year,
            s.cip2,
            s.cip4,
            s.cip_code,
            s.employer,
            s.weighted_n AS employer_weighted_n,
            s.n AS employer_n,
            s.unknown_employer_n,
            s.named_employer_n,
            s.same_school_employer_n,
            s.career_employer_n,
            ROW_NUMBER() OVER (
                PARTITION BY s.unitid, s.degree, s.horizon, s.cohort_year, s.cip2, s.cip4, s.cip_code
                ORDER BY s.weighted_n DESC, s.employer
            ) AS employer_rank
        FROM (
            SELECT * FROM ({employer_by_major})
            UNION ALL
            SELECT * FROM ({employer_all_major})
        ) s
    ),
    role_rows AS (
        SELECT * FROM ({role_by_major})
        UNION ALL
        SELECT * FROM ({role_all_major})
    ),
    ranked AS (
        SELECT
            r.*,
            e.employer_n AS employer_total_n,
            e.unknown_employer_n AS employer_unknown_employer_n,
            e.named_employer_n AS employer_named_employer_n,
            e.same_school_employer_n AS employer_same_school_employer_n,
            e.career_employer_n AS employer_career_employer_n,
            e.employer_rank,
            ROUND(r.weighted_n / NULLIF(e.employer_weighted_n, 0) * 100, 2) AS role_share_pct,
            ROUND(r.weighted_n / NULLIF(e.career_employer_n, 0) * 100, 2) AS role_share_pct_career,
            ROW_NUMBER() OVER (
                PARTITION BY r.unitid, r.degree, r.horizon, r.cohort_year, r.cip2, r.cip4, r.cip_code, r.employer
                ORDER BY r.weighted_n DESC, r.role_k50_v3, r.role_k150_v3
            ) AS role_rank
        FROM role_rows r
        JOIN employer_rows e
          ON r.unitid = e.unitid
         AND r.degree = e.degree
         AND r.horizon = e.horizon
         AND r.cohort_year = e.cohort_year
         AND r.cip2 = e.cip2
         AND r.cip4 = e.cip4
         AND r.cip_code = e.cip_code
         AND r.employer = e.employer
    )
    SELECT * FROM ranked
""")
employer_role.columns = [c.lower() for c in employer_role.columns]

print(f'school_employer_role_fact: {len(employer_role):,} rows')
write_fact_parquet(employer_role, OUT_DIR / 'school_employer_role_fact.parquet')
print('\nSaved school_employer_role_fact.parquet')


school_employer_fact: 2,286,900 rows

Top employers at Berkeley (Bachelors, 5yr, 2018, all majors):
                employer   n share_pct share_pct_career  same_school_employer_n  unknown_employer_n
                 Unknown 519      8.89            10.21                       0                 519
University of California 234      4.01             4.60                     234                   0
          Alphabet, Inc. 152      2.60             2.99                       0                   0
        Amazon.com, Inc.  97      1.66             1.91                       0                   0
    Meta Platforms, Inc.  79      1.35             1.55                       0                   0
             Apple, Inc.  55      0.94             1.08                       0                   0
         Microsoft Corp.  43      0.74             0.85                       0                   0
     Stanford University  33      0.57             0.65                       0                   0


school_employer_role_fact: 2,953,132 rows

Sample employer-role rows at Berkeley (Bachelors, 5yr, 2018, all majors):
                                    employer           role_k50_v3  n role_share_pct role_share_pct_career  same_school_employer_n
Oakland Unified School District (California)               Teacher  4          66.67                 66.67                       0
Oakland Unified School District (California)     Change Consultant  1          16.67                 16.67                       0
Oakland Unified School District (California)      Research Scholar  1          16.67                 16.67                       0
                       Schneider Electric SE              Engineer  1          50.00                 50.00                       0
                       Schneider Electric SE    Software Developer  1          50.00                 50.00                       0
                       Ares Management Corp.       Banking Analyst  1          50.00             

In [6]:
# =============================================================
# CELL 5: weighted school_geo_fact
# =============================================================
# Geography outcomes by school x degree x horizon x cohort x major.
# Includes ALL-major rows for school-level views.

geo_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip4 AS cip_code, cip_title,
        COALESCE(metro_area, state, country, 'Unknown') AS location,
        state, country, salary, seniority, analysis_weight
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
"""
geo_by_major = weighted_aggregate_sql(
    geo_source + "\nWHERE cip4 IS NOT NULL",
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'location', 'state', 'country'],
    count_alias='n',
    salary_count_alias='salary_obs',
)
geo_all_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        'ALL' AS cip2, 'ALL' AS cip4, 'ALL' AS cip_code, 'All majors' AS cip_title,
        COALESCE(metro_area, state, country, 'Unknown') AS location,
        state, country, salary, seniority, analysis_weight
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
"""
geo_all_major = weighted_aggregate_sql(
    geo_all_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'location', 'state', 'country'],
    count_alias='n',
    salary_count_alias='salary_obs',
)
geo = sfClient.load_df(f"""
    WITH stacked AS (
        SELECT * FROM ({geo_by_major})
        UNION ALL
        SELECT * FROM ({geo_all_major})
    ),
    totals AS (
        SELECT unitid, degree, horizon, cohort_year, cip2, cip4, cip_code, SUM(weighted_n) AS total_weighted_n
        FROM stacked
        GROUP BY unitid, degree, horizon, cohort_year, cip2, cip4, cip_code
    ),
    ranked AS (
        SELECT
            s.*,
            ROUND(s.weighted_n / NULLIF(t.total_weighted_n, 0) * 100, 2) AS share_pct,
            ROW_NUMBER() OVER (
                PARTITION BY s.unitid, s.degree, s.horizon, s.cohort_year, s.cip2, s.cip4, s.cip_code
                ORDER BY s.weighted_n DESC, s.location
            ) AS geo_rank
        FROM stacked s
        JOIN totals t
          ON s.unitid = t.unitid
         AND s.degree = t.degree
         AND s.horizon = t.horizon
         AND s.cohort_year = t.cohort_year
         AND s.cip2 = t.cip2
         AND s.cip4 = t.cip4
         AND s.cip_code = t.cip_code
    )
    SELECT * FROM ranked
""")
geo.columns = [c.lower() for c in geo.columns]

print(f'school_geo_fact: {len(geo):,} rows')
write_fact_parquet(geo, OUT_DIR / 'school_geo_fact.parquet')
print('\nSaved.')


school_geo_fact: 919,093 rows

Top locations for Berkeley (Bachelors, 5yr, 2018, all majors):
                               location    n share_pct  median_salary
        san francisco metropolitan area 1960     33.57       122463.0
          los angeles metropolitan area  569      9.74        86873.0
san jose metropolitan area (california)  521      8.92       148631.0
        new york city metropolitan area  359      6.15       143246.0
                                  empty  233      3.99        97303.0
            san diego metropolitan area  158      2.71        98966.0
              seattle metropolitan area  132      2.26       136287.0
              anaheim metropolitan area  126      2.16        94163.0
        california nonmetropolitan area  115      1.97        78566.0
                                  empty  107      1.83        99476.0

Saved.


In [7]:
# =============================================================
# CELL 6: weighted school_role_fact
# =============================================================
# Role and industry hierarchy by school x degree x horizon x cohort x major.
# Includes ALL-major rows so school-wide and major-specific treemaps share one fact.

role_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip4 AS cip_code, cip_title,
        role_k10_v3, role_k50_v3, role_k150_v3,
        rics_k50 AS industry_k50,
        rics_k200 AS industry_k200,
        salary, seniority, analysis_weight
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
"""
role_by_major = weighted_aggregate_sql(
    role_source + "\nWHERE cip4 IS NOT NULL",
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'role_k10_v3', 'role_k50_v3', 'role_k150_v3', 'industry_k50', 'industry_k200'],
    count_alias='n',
    salary_count_alias='salary_obs',
)
role_all_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        'ALL' AS cip2, 'ALL' AS cip4, 'ALL' AS cip_code, 'All majors' AS cip_title,
        role_k10_v3, role_k50_v3, role_k150_v3,
        rics_k50 AS industry_k50,
        rics_k200 AS industry_k200,
        salary, seniority, analysis_weight
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
"""
role_all_major = weighted_aggregate_sql(
    role_all_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'role_k10_v3', 'role_k50_v3', 'role_k150_v3', 'industry_k50', 'industry_k200'],
    count_alias='n',
    salary_count_alias='salary_obs',
)
role = sfClient.load_df(f"""
    WITH stacked AS (
        SELECT * FROM ({role_by_major})
        UNION ALL
        SELECT * FROM ({role_all_major})
    ),
    totals AS (
        SELECT unitid, degree, horizon, cohort_year, cip2, cip4, cip_code, SUM(weighted_n) AS total_weighted_n
        FROM stacked
        GROUP BY unitid, degree, horizon, cohort_year, cip2, cip4, cip_code
    )
    SELECT
        s.*,
        ROUND(s.weighted_n / NULLIF(t.total_weighted_n, 0) * 100, 2) AS share_pct
    FROM stacked s
    JOIN totals t
      ON s.unitid = t.unitid
     AND s.degree = t.degree
     AND s.horizon = t.horizon
     AND s.cohort_year = t.cohort_year
     AND s.cip2 = t.cip2
     AND s.cip4 = t.cip4
     AND s.cip_code = t.cip_code
""")
role.columns = [c.lower() for c in role.columns]

print(f'school_role_fact: {len(role):,} rows')
write_fact_parquet(role, OUT_DIR / 'school_role_fact.parquet')
print('\nSaved.')


school_role_fact: 2,090,757 rows

Top role_k10 at Berkeley (Bachelors, 5yr, 2018, all majors):
                                 n  median_salary
role_k10_v3                                      
Public Service and Education  1252   89008.570248
Software Engineer             1227  140700.305344
Sales and Marketing            984  105801.928753
Finance                        653  111117.242537
Engineer                       546  110351.800000
Project and IT Specialist      372  114397.629630
Healthcare Provider            285   96194.504505
Operations                     248   94773.486188
Service Worker                 204   76269.122951
Technician                      68   92453.509434



Saved.


In [8]:
# =============================================================
# CELL 7: weighted school_major_fact + school_major_mix_fact
# =============================================================
# Outcomes and composition by school x degree x horizon x cohort x major.
# CIP4 is the primary major key; CIP6 remains diagnostic in the earnings fact.

major_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip4 AS cip_code, cip_title,
        salary, seniority, analysis_weight,
        ultimate_parent_company_name, company_name, metro_area, state, country,
        cip_probability, high_conf_major_flag
    FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE
    WHERE cip4 IS NOT NULL
"""
major_counts_sql = weighted_aggregate_sql(
    major_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title'],
    count_alias='n',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "COUNT(DISTINCT CASE WHEN LOWER(COALESCE(ultimate_parent_company_name, '')) = 'government of the united states of america' AND NULLIF(company_name, '') IS NOT NULL THEN company_name ELSE COALESCE(ultimate_parent_company_name, company_name) END) AS unique_employers",
        "COUNT(DISTINCT COALESCE(metro_area, state, country)) AS unique_metros",
        "ROUND(SUM(CASE WHEN cip_probability IS NOT NULL THEN cip_probability * COALESCE(analysis_weight, 1.0) ELSE 0 END) / NULLIF(SUM(CASE WHEN cip_probability IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END), 0), 3) AS avg_cip_prob",
        "ROUND(SUM(COALESCE(high_conf_major_flag, 0) * COALESCE(analysis_weight, 1.0)), 0) AS high_conf_major_n",
    ],
)
major = sfClient.load_df(f"""
    WITH major_counts AS (
        SELECT * FROM ({major_counts_sql})
    ),
    totals AS (
        SELECT unitid, degree, horizon, cohort_year, SUM(weighted_n) AS total_weighted_n
        FROM major_counts
        GROUP BY unitid, degree, horizon, cohort_year
    ),
    ranked AS (
        SELECT
            m.*,
            ROUND(m.weighted_n / NULLIF(t.total_weighted_n, 0) * 100, 2) AS share_pct,
            ROW_NUMBER() OVER (
                PARTITION BY m.unitid, m.degree, m.horizon, m.cohort_year
                ORDER BY m.weighted_n DESC, m.cip_code
            ) AS major_rank
        FROM major_counts m
        JOIN totals t
          ON m.unitid = t.unitid
         AND m.degree = t.degree
         AND m.horizon = t.horizon
         AND m.cohort_year = t.cohort_year
    )
    SELECT * FROM ranked
""")
major.columns = [c.lower() for c in major.columns]
major['current_student_flag'] = 0

major_mix = major[[
    'unitid','ipeds_name','degree','horizon','cohort_year','cohort_band',
    'cip2','cip4','cip_code','cip_title','n','share_pct','avg_cip_prob','high_conf_major_n','major_rank','current_student_flag'
]].copy()

# Add projected current-student major composition for future undergraduate classes.
# This feeds over-time major growth only and does not affect earnings or employer facts.
if 'startdate' in EDUCATION_COLUMNS:
    horizon_sql = " UNION ALL ".join(f"SELECT {h} AS horizon" for h in HORIZONS)
    current_major_mix = sfClient.load_df(f"""
        WITH current_undergrads AS (
            SELECT
                e.user_id,
                e.unitid,
                e.ipeds_name,
                'Bachelors' AS degree,
                CASE
                    WHEN e.enddate IS NOT NULL AND YEAR(e.enddate) BETWEEN 2026 AND 2029 THEN YEAR(e.enddate)
                    WHEN e.enddate IS NULL AND e.startdate IS NOT NULL AND YEAR(e.startdate) BETWEEN 2022 AND 2025 THEN YEAR(e.startdate) + 4
                END AS cohort_year,
                '2026-2029' AS cohort_band,
                LEFT({assigned_cip4_sql('e')}, 2) AS cip2,
                {assigned_cip4_sql('e')} AS cip4,
                {assigned_cip4_sql('e')} AS cip_code,
                COALESCE(c4.title, {assigned_cip_title_sql('e')}, '') AS cip_title,
                e.cip_probability,
                CASE WHEN e.cip_probability >= 0.8 THEN 1 ELSE 0 END AS high_conf_major_flag
            FROM {EDUCATION_CIP} e
            LEFT JOIN {SCRATCH}.CIP4_TITLES c4
              ON {assigned_cip4_sql('e')} = c4.code
            WHERE e.unitid IN ({UNITID_SQL})
              AND e.degree = 'Bachelor'
              AND {assigned_cip4_sql('e')} IS NOT NULL
              AND (
                  (e.enddate IS NOT NULL AND YEAR(e.enddate) BETWEEN 2026 AND 2029)
                  OR
                  (e.enddate IS NULL AND e.startdate IS NOT NULL AND YEAR(e.startdate) BETWEEN 2022 AND 2025)
              )
        ),
        projected AS (
            SELECT *
            FROM current_undergrads
            WHERE cohort_year BETWEEN 2026 AND 2029
        ),
        horizons AS (
            {horizon_sql}
        ),
        major_counts AS (
            SELECT
                p.unitid,
                p.ipeds_name,
                p.degree,
                h.horizon,
                p.cohort_year,
                p.cohort_band,
                p.cip2,
                p.cip4,
                p.cip_code,
                MAX(p.cip_title) AS cip_title,
                COUNT(DISTINCT p.user_id) AS n,
                ROUND(AVG(p.cip_probability), 3) AS avg_cip_prob,
                COUNT(DISTINCT CASE WHEN p.high_conf_major_flag = 1 THEN p.user_id END) AS high_conf_major_n
            FROM projected p
            CROSS JOIN horizons h
            GROUP BY p.unitid, p.ipeds_name, p.degree, h.horizon, p.cohort_year, p.cohort_band, p.cip2, p.cip4, p.cip_code
            HAVING COUNT(DISTINCT p.user_id) >= 1
        ),
        totals AS (
            SELECT unitid, degree, horizon, cohort_year, SUM(n) AS total_n
            FROM major_counts
            GROUP BY unitid, degree, horizon, cohort_year
        ),
        ranked AS (
            SELECT
                m.*,
                ROUND(m.n / t.total_n * 100, 2) AS share_pct,
                ROW_NUMBER() OVER (
                    PARTITION BY m.unitid, m.degree, m.horizon, m.cohort_year
                    ORDER BY m.n DESC, m.cip_code
                ) AS major_rank,
                1 AS current_student_flag
            FROM major_counts m
            JOIN totals t
              ON m.unitid = t.unitid
             AND m.degree = t.degree
             AND m.horizon = t.horizon
             AND m.cohort_year = t.cohort_year
        )
        SELECT * FROM ranked
    """)
    current_major_mix.columns = [c.lower() for c in current_major_mix.columns]
    current_major_mix['unitid'] = current_major_mix['unitid'].astype(str)
    major_mix = pd.concat([major_mix, current_major_mix[major_mix.columns]], ignore_index=True)
    print(f'Added {len(current_major_mix):,} projected current-student major_mix rows for classes 2026-2029.')
else:
    print('Skipping projected current-student major_mix rows because startdate is unavailable.')

print(f'school_major_fact: {len(major):,} rows')
print(f'school_major_mix_fact: {len(major_mix):,} rows')
write_fact_parquet(major, OUT_DIR / 'school_major_fact.parquet')
write_fact_parquet(major_mix, OUT_DIR / 'school_major_mix_fact.parquet')
print('\nSaved school_major_fact.parquet and school_major_mix_fact.parquet')


school_major_fact: 77,070 rows

Top majors at Berkeley (Bachelors, 5yr, 2018):
cip_code                                       cip_title   n share_pct  median_salary
 11.0701                                Computer Science 828     15.97       180741.0
 45.0601                              Economics, General 506      9.76       117385.0
 52.0201 Business Administration and Management, General 354      6.83       120286.0
 45.1001       Political Science and Government, General 307      5.92        89612.0
 26.0406             Cell/Cellular and Molecular Biology 304      5.86        73251.0
 09.0102                Mass Communication/Media Studies 170      3.28        83605.0
 45.1101                              Sociology, General 169      3.26        85211.0
 23.0101        English Language and Literature, General 148      2.85        71076.0
 30.2501                      Cognitive Science, General 147      2.83       122547.0
 14.1901                          Mechanical Engineering 142 

In [9]:
# =============================================================
# CELL 8: weighted school_postgrad_fact + destination/degree flow facts
# =============================================================
# Bachelor's grads are linked to later education using user_id.
# Counts use the IPEDS-derived education calibration weight only; position
# weights do not apply because these are education-to-education flows.

postgrad = sfClient.load_df(f"""
    WITH bachelors AS (
        SELECT
            user_id,
            unitid,
            ipeds_name,
            cohort_year,
            cohort_band,
            cip2 AS undergrad_cip2,
            cip4 AS undergrad_cip4,
            cip_code AS undergrad_cip_code,
            cip_title AS undergrad_cip_title,
            grad_date,
            COALESCE(ipeds_calibration_weight, 1.0) AS education_weight
        FROM {SCRATCH}.SCHOOL_GRADS_CALIBRATED
        WHERE degree = 'Bachelors'
    ),
    later_edu AS (
        SELECT
            b.user_id,
            b.unitid,
            b.cohort_year,
            b.cohort_band,
            b.undergrad_cip2,
            b.undergrad_cip4,
            b.undergrad_cip_code,
            {postgrad_degree_label_sql('e2')} AS postgrad_degree,
            e2.ipeds_name AS postgrad_school,
            LEFT({assigned_cip4_sql('e2')}, 2) AS postgrad_cip2,
            {assigned_cip4_sql('e2')} AS postgrad_cip4,
            {assigned_cip4_sql('e2')} AS postgrad_cip_code,
            {assigned_cip_title_sql('e2')} AS postgrad_cip_title,
            YEAR(e2.enddate) AS postgrad_year,
            DATEDIFF('day', b.grad_date, e2.enddate) / 365.25 AS years_to_postgrad,
            ROW_NUMBER() OVER (
                PARTITION BY b.user_id, b.unitid, b.cohort_year
                ORDER BY e2.enddate ASC
            ) AS edu_rank
        FROM bachelors b
        JOIN {EDUCATION_CIP} e2
          ON b.user_id = e2.user_id
         AND (e2.degree IN ('Master', 'MBA') OR e2.degree LIKE 'Doctor%')
         AND e2.enddate > b.grad_date
    )
    SELECT
        b.unitid,
        MAX(b.ipeds_name) AS ipeds_name,
        b.cohort_year,
        b.cohort_band,
        b.undergrad_cip2,
        b.undergrad_cip4,
        b.undergrad_cip_code,
        MAX(b.undergrad_cip_title) AS undergrad_cip_title,
        ROUND(SUM(b.education_weight), 0) AS total_bachelors,
        COUNT(DISTINCT b.user_id) AS raw_total_bachelors,
        ROUND(SUM(CASE WHEN le.user_id IS NOT NULL THEN b.education_weight ELSE 0 END), 0) AS has_later_degree,
        COUNT(DISTINCT CASE WHEN le.user_id IS NOT NULL THEN b.user_id END) AS raw_has_later_degree,
        ROUND(SUM(CASE WHEN le.postgrad_degree = 'Masters' THEN b.education_weight ELSE 0 END), 0) AS masters_count,
        ROUND(SUM(CASE WHEN le.postgrad_degree = 'MBA' THEN b.education_weight ELSE 0 END), 0) AS mba_count,
        ROUND(SUM(CASE WHEN le.postgrad_degree = 'LAW' THEN b.education_weight ELSE 0 END), 0) AS law_count,
        ROUND(SUM(CASE WHEN le.postgrad_degree = 'MD' THEN b.education_weight ELSE 0 END), 0) AS md_count,
        ROUND(SUM(CASE WHEN le.postgrad_degree = 'PhD' THEN b.education_weight ELSE 0 END), 0) AS phd_count,
        ROUND(SUM(CASE WHEN le.postgrad_degree = 'Professional Doctorate' THEN b.education_weight ELSE 0 END), 0) AS professional_doctorate_count,
        ROUND(SUM(CASE WHEN le.postgrad_degree = 'Other Doctorate' THEN b.education_weight ELSE 0 END), 0) AS other_doctorate_count,
        ROUND(SUM(CASE WHEN le.postgrad_degree IN ('PhD', 'LAW', 'MD', 'Professional Doctorate', 'Other Doctorate', 'Doctorate') THEN b.education_weight ELSE 0 END), 0) AS doctor_count
    FROM bachelors b
    LEFT JOIN later_edu le
      ON b.user_id = le.user_id
     AND b.unitid = le.unitid
     AND b.cohort_year = le.cohort_year
     AND le.edu_rank = 1
    GROUP BY b.unitid, b.cohort_year, b.cohort_band,
             b.undergrad_cip2, b.undergrad_cip4, b.undergrad_cip_code
    HAVING total_bachelors >= 1
""")
postgrad.columns = [c.lower() for c in postgrad.columns]
postgrad['later_degree_pct'] = (100 * postgrad['has_later_degree'] / postgrad['total_bachelors']).round(1)

postgrad_flow = sfClient.load_df(f"""
    WITH bachelors AS (
        SELECT
            user_id,
            unitid,
            ipeds_name,
            cohort_year,
            cohort_band,
            cip2 AS undergrad_cip2,
            cip4 AS undergrad_cip4,
            cip_code AS undergrad_cip_code,
            cip_title AS undergrad_cip_title,
            grad_date,
            COALESCE(ipeds_calibration_weight, 1.0) AS education_weight
        FROM {SCRATCH}.SCHOOL_GRADS_CALIBRATED
        WHERE degree = 'Bachelors'
    ),
    later_edu AS (
        SELECT
            b.*,
            {postgrad_degree_label_sql('e2')} AS postgrad_degree,
            YEAR(e2.enddate) AS postgrad_year,
            DATEDIFF('day', b.grad_date, e2.enddate) / 365.25 AS years_to_postgrad,
            ROW_NUMBER() OVER (
                PARTITION BY b.user_id, b.unitid, b.cohort_year
                ORDER BY e2.enddate ASC
            ) AS edu_rank
        FROM bachelors b
        JOIN {EDUCATION_CIP} e2
          ON b.user_id = e2.user_id
         AND (e2.degree IN ('Master', 'MBA') OR e2.degree LIKE 'Doctor%')
         AND e2.enddate > b.grad_date
    ),
    first_flow AS (
        SELECT * FROM later_edu WHERE edu_rank = 1
    ),
    base_totals AS (
        SELECT
            unitid,
            cohort_year,
            undergrad_cip2,
            undergrad_cip4,
            undergrad_cip_code,
            SUM(education_weight) AS weighted_total_bachelors,
            COUNT(DISTINCT user_id) AS raw_total_bachelors
        FROM bachelors
        GROUP BY unitid, cohort_year, undergrad_cip2, undergrad_cip4, undergrad_cip_code
    )
    SELECT
        f.unitid,
        f.ipeds_name,
        f.cohort_year,
        f.cohort_band,
        f.undergrad_cip2,
        f.undergrad_cip4,
        f.undergrad_cip_code,
        f.undergrad_cip_title,
        f.postgrad_degree,
        ROUND(SUM(f.education_weight), 0) AS n_users,
        COUNT(DISTINCT f.user_id) AS raw_n_users,
        ROUND(SUM(f.years_to_postgrad * f.education_weight) / NULLIF(SUM(f.education_weight), 0), 2) AS avg_years_to_postgrad,
        ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY f.years_to_postgrad), 2) AS median_years_to_postgrad,
        ROUND(t.weighted_total_bachelors, 0) AS total_bachelors,
        t.raw_total_bachelors,
        ROUND(SUM(f.education_weight) / NULLIF(t.weighted_total_bachelors, 0) * 100, 2) AS flow_pct
    FROM first_flow f
    JOIN base_totals t
      ON f.unitid = t.unitid
     AND f.cohort_year = t.cohort_year
     AND f.undergrad_cip2 = t.undergrad_cip2
     AND f.undergrad_cip4 = t.undergrad_cip4
     AND f.undergrad_cip_code = t.undergrad_cip_code
    GROUP BY f.unitid, f.ipeds_name, f.cohort_year, f.cohort_band,
             f.undergrad_cip2, f.undergrad_cip4, f.undergrad_cip_code, f.undergrad_cip_title,
             f.postgrad_degree, t.weighted_total_bachelors, t.raw_total_bachelors
    HAVING n_users >= 1
""")
postgrad_flow.columns = [c.lower() for c in postgrad_flow.columns]

postgrad_dest = sfClient.load_df(f"""
    WITH bachelors AS (
        SELECT
            user_id,
            unitid,
            ipeds_name,
            cohort_year,
            cohort_band,
            cip2 AS undergrad_cip2,
            cip4 AS undergrad_cip4,
            cip_code AS undergrad_cip_code,
            cip_title AS undergrad_cip_title,
            grad_date,
            COALESCE(ipeds_calibration_weight, 1.0) AS education_weight
        FROM {SCRATCH}.SCHOOL_GRADS_CALIBRATED
        WHERE degree = 'Bachelors'
    ),
    later_edu AS (
        SELECT
            b.*,
            {postgrad_degree_label_sql('e2')} AS postgrad_degree,
            e2.ipeds_name AS postgrad_school,
            LEFT({assigned_cip4_sql('e2')}, 2) AS postgrad_cip2,
            {assigned_cip4_sql('e2')} AS postgrad_cip4,
            {assigned_cip4_sql('e2')} AS postgrad_cip_code,
            {assigned_cip_title_sql('e2')} AS postgrad_cip_title,
            YEAR(e2.enddate) AS postgrad_year,
            DATEDIFF('day', b.grad_date, e2.enddate) / 365.25 AS years_to_postgrad,
            ROW_NUMBER() OVER (
                PARTITION BY b.user_id, b.unitid, b.cohort_year
                ORDER BY e2.enddate ASC
            ) AS edu_rank
        FROM bachelors b
        JOIN {EDUCATION_CIP} e2
          ON b.user_id = e2.user_id
         AND (e2.degree IN ('Master', 'MBA') OR e2.degree LIKE 'Doctor%')
         AND e2.enddate > b.grad_date
    )
    SELECT
        le.unitid,
        le.ipeds_name,
        le.cohort_year,
        le.cohort_band,
        le.undergrad_cip2,
        le.undergrad_cip4,
        le.undergrad_cip_code,
        le.undergrad_cip_title,
        le.postgrad_degree,
        le.postgrad_school,
        le.postgrad_cip2,
        le.postgrad_cip4,
        le.postgrad_cip_code,
        le.postgrad_cip_title,
        ROUND(SUM(le.education_weight), 0) AS n,
        COUNT(DISTINCT le.user_id) AS raw_n,
        ROUND(SUM(le.years_to_postgrad * le.education_weight) / NULLIF(SUM(le.education_weight), 0), 2) AS avg_years_to_postgrad
    FROM later_edu le
    WHERE le.edu_rank = 1
    GROUP BY le.unitid, le.ipeds_name, le.cohort_year, le.cohort_band,
             le.undergrad_cip2, le.undergrad_cip4, le.undergrad_cip_code, le.undergrad_cip_title,
             le.postgrad_degree, le.postgrad_school, le.postgrad_cip2, le.postgrad_cip4, le.postgrad_cip_code, le.postgrad_cip_title
    HAVING n >= 1
""")
postgrad_dest.columns = [c.lower() for c in postgrad_dest.columns]

print(f'school_postgrad_fact: {len(postgrad):,} rows')
print(f'school_postgrad_flow_fact: {len(postgrad_flow):,} rows')
print(f'school_postgrad_destination_fact: {len(postgrad_dest):,} rows')

write_fact_parquet(postgrad, OUT_DIR / 'school_postgrad_fact.parquet')
write_fact_parquet(postgrad_flow, OUT_DIR / 'school_postgrad_flow_fact.parquet')
write_fact_parquet(postgrad_dest, OUT_DIR / 'school_postgrad_destination_fact.parquet')
print('\nSaved postgrad summary, degree flow, and destination facts.')


school_postgrad_fact: 10,311 rows
school_postgrad_flow_fact: 22,150 rows
school_postgrad_destination_fact: 150,477 rows

Harvard bachelor degree-flow sample:
 cohort_year undergrad_cip_code                            undergrad_cip_title postgrad_degree  n_users flow_pct median_years_to_postgrad
        2005            14.1401 Environmental/Environmental Health Engineering         Masters        1   100.00                     3.00
        2005            16.0901                 French Language and Literature         Masters        1   100.00                     4.00
        2005            05.0201                 African-American/Black Studies         Masters        1   100.00                     5.00
        2005            26.0301                           Botany/Plant Biology         Masters        2   100.00                     6.50
        2005            16.0905                Spanish Language and Literature         Masters        1   100.00                     5.00
        2005  

In [10]:
# =============================================================
# CELL 9: school_coverage_fact
# =============================================================
# Coverage and match quality by school x degree x cohort x CIP4.
# Builds from an IPEDS scaffold first, then left/outer joins observed
# Revelio rows so zero-observation IPEDS majors remain visible.

degree_map = pd.DataFrame([
    ('01', 'Associates'), ('1', 'Associates'), ('02', 'Associates'), ('2', 'Associates'), ('03', 'Associates'), ('3', 'Associates'), ('04', 'Associates'), ('4', 'Associates'),
    ('05', 'Bachelors'), ('5', 'Bachelors'),
    ('07', 'Masters'), ('7', 'Masters'),
    ('17', 'Research Doctorate'),
    ('18', 'Professional Doctorate'), ('09', 'Professional Doctorate'), ('9', 'Professional Doctorate'), ('10', 'Professional Doctorate'), ('11', 'Professional Doctorate'),
    ('19', 'Other Doctorate'),
], columns=['awlevel', 'ipeds_degree_level'])

coverage_obs = sfClient.load_df(f"""
    WITH base AS (
        SELECT
            e.user_id,
            e.unitid,
            e.ipeds_name,
            {degree_label_sql('e')} AS degree,
            {ipeds_degree_level_sql('e')} AS ipeds_degree_level,
            YEAR(e.enddate) AS cohort_year,
            CASE
                WHEN YEAR(e.enddate) BETWEEN 2005 AND 2009 THEN '2005-2009'
                WHEN YEAR(e.enddate) BETWEEN 2010 AND 2014 THEN '2010-2014'
                WHEN YEAR(e.enddate) BETWEEN 2015 AND 2019 THEN '2015-2019'
                WHEN YEAR(e.enddate) BETWEEN 2020 AND 2025 THEN '2020-2025'
            END AS cohort_band,
            LEFT({assigned_cip4_sql('e')}, 2) AS cip2,
            {assigned_cip4_sql('e')} AS cip4,
            {assigned_cip4_sql('e')} AS cip_code,
            COALESCE(c4.title, {assigned_cip_title_sql('e')}, '') AS cip_title,
            e.cip_probability,
            e.match_source,
            e.cip_match_type
        FROM {EDUCATION_CIP} e
        LEFT JOIN {SCRATCH}.CIP4_TITLES c4
          ON {assigned_cip4_sql('e')} = c4.code
        WHERE e.unitid IN ({UNITID_SQL})
          AND (e.degree IN ('Associate', 'Bachelor', 'Master', 'MBA') OR e.degree = 'Doctor')
          AND e.enddate IS NOT NULL
          AND YEAR(e.enddate) BETWEEN 2005 AND 2025
    )
    SELECT
        unitid,
        ipeds_name,
        degree,
        ipeds_degree_level,
        cohort_year,
        cohort_band,
        cip2,
        cip4,
        cip_code,
        MAX(cip_title) AS cip_title,
        COUNT(*) AS raw_education_rows,
        COUNT(DISTINCT user_id) AS total_records,
        COUNT(DISTINCT CASE WHEN cip_code IS NOT NULL THEN user_id END) AS has_cip,
        ROUND(AVG(cip_probability), 3) AS avg_cip_prob,
        COUNT(DISTINCT CASE WHEN match_source = 'triple' THEN user_id END) AS triple_matches,
        COUNT(DISTINCT CASE WHEN match_source = 'manual' THEN user_id END) AS manual_matches,
        COUNT(DISTINCT CASE WHEN cip_match_type = 'school_year_level' THEN user_id END) AS school_year_matches,
        COUNT(DISTINCT CASE WHEN cip_probability >= 0.8 THEN user_id END) AS high_confidence
    FROM base
    WHERE cip_code IS NOT NULL
    GROUP BY unitid, ipeds_name, degree, ipeds_degree_level, cohort_year, cohort_band, cip2, cip4, cip_code
    HAVING total_records >= 1
""")
coverage_obs.columns = [c.lower() for c in coverage_obs.columns]
coverage_obs['unitid'] = coverage_obs['unitid'].astype(str)

ipeds = None
ipeds_source = None

try:
    sfClient.load_df(f"SELECT 1 AS ok FROM {IPEDS_COMPLETIONS_TABLE} LIMIT 1")
    ipeds = sfClient.load_df(f"""
        SELECT
            CAST(c.unitid AS VARCHAR) AS unitid,
            dm.ipeds_degree_level,
            c.year AS cohort_year,
            CASE
                WHEN c.year BETWEEN 2005 AND 2009 THEN '2005-2009'
                WHEN c.year BETWEEN 2010 AND 2014 THEN '2010-2014'
                WHEN c.year BETWEEN 2015 AND 2019 THEN '2015-2019'
                WHEN c.year BETWEEN 2020 AND 2025 THEN '2020-2025'
            END AS cohort_band,
            LEFT(c.cipcode, 2) AS cip2,
            CASE WHEN c.cipcode IS NOT NULL THEN SUBSTR(c.cipcode, 1, 5) END AS cip4,
            CASE WHEN c.cipcode IS NOT NULL THEN SUBSTR(c.cipcode, 1, 5) END AS cip_code,
            SUM(c.ctotalt) AS ipeds_completions
        FROM {IPEDS_COMPLETIONS_TABLE} c
        JOIN (SELECT column1 AS awlevel, column2 AS ipeds_degree_level FROM VALUES
            ('01', 'Associates'), ('1', 'Associates'), ('02', 'Associates'), ('2', 'Associates'), ('03', 'Associates'), ('3', 'Associates'), ('04', 'Associates'), ('4', 'Associates'),
            ('05', 'Bachelors'), ('5', 'Bachelors'),
            ('07', 'Masters'), ('7', 'Masters'),
            ('17', 'Research Doctorate'),
            ('18', 'Professional Doctorate'), ('09', 'Professional Doctorate'), ('9', 'Professional Doctorate'), ('10', 'Professional Doctorate'), ('11', 'Professional Doctorate'),
            ('19', 'Other Doctorate')
        ) dm
          ON CAST(c.awlevel AS VARCHAR) = dm.awlevel
        WHERE CAST(c.unitid AS VARCHAR) IN ({UNITID_SQL})
          AND c.year BETWEEN 2005 AND 2025
          AND c.cipcode IS NOT NULL
        GROUP BY CAST(c.unitid AS VARCHAR), dm.ipeds_degree_level, c.year, cohort_band, cip2, cip4, cip_code
    """)
    ipeds.columns = [c.lower() for c in ipeds.columns]
    ipeds['unitid'] = ipeds['unitid'].astype(str)
    ipeds_source = f'snowflake table {IPEDS_COMPLETIONS_TABLE}'
except Exception as err:
    print(f'IPEDS table unavailable; falling back to local comp build: {err}')
    if 'get_local_ipeds_comp' in globals():
        _, comp_local = get_local_ipeds_comp()
    elif 'comp' in globals():
        comp_local = comp.copy()
    else:
        raise RuntimeError('No IPEDS completion source available.')

    ipeds = comp_local.copy()
    ipeds['unitid'] = ipeds['unitid'].astype(str)
    ipeds['cipcode'] = ipeds['cipcode'].astype(str).str.replace(r'^="?|"$', '', regex=True)
    ipeds['awlevel'] = ipeds['awlevel'].astype(str)
    ipeds['year'] = pd.to_numeric(ipeds['year'], errors='coerce')
    ipeds = ipeds[
        ipeds['unitid'].isin(set(UNITID_LIST))
        & ipeds['year'].between(2005, 2025)
        & ipeds['cipcode'].notna()
        & (ipeds['cipcode'] != '')
    ].copy()
    ipeds = ipeds.merge(degree_map, on='awlevel', how='inner')
    ipeds['cohort_year'] = ipeds['year'].astype(int)
    ipeds['cohort_band'] = np.select(
        [
            ipeds['cohort_year'].between(2005, 2009),
            ipeds['cohort_year'].between(2010, 2014),
            ipeds['cohort_year'].between(2015, 2019),
            ipeds['cohort_year'].between(2020, 2025),
        ],
        ['2005-2009', '2010-2014', '2015-2019', '2020-2025'],
        default=None,
    )
    ipeds['cip2'] = ipeds['cipcode'].str[:2]
    ipeds['cip4'] = np.where(ipeds['cipcode'].notna(), ipeds['cipcode'].str[:5], None)
    ipeds = (
        ipeds.groupby(['unitid', 'ipeds_degree_level', 'cohort_year', 'cohort_band', 'cip2', 'cip4'], as_index=False)['ctotalt']
        .sum()
        .rename(columns={'ctotalt': 'ipeds_completions'})
    )
    ipeds['cip_code'] = ipeds['cip4']
    ipeds_source = 'local IPEDS completions files (_A)'

print(f'Using {ipeds_source} for IPEDS denominators.')

school_name_lookup = school_meta.assign(unitid=school_meta['unitid'].astype(str)).set_index('unitid')['ipeds_name'].astype(str).to_dict()
cip4_title_lookup = cip4_df.set_index('cip4')['cip4_title'].astype(str).to_dict()

coverage = ipeds.merge(
    coverage_obs,
    on=['unitid', 'ipeds_degree_level', 'cohort_year', 'cip_code'],
    how='outer',
    suffixes=('_ipeds', '_obs'),
    indicator=True,
)

coverage['ipeds_name'] = coverage['ipeds_name'].fillna(coverage['unitid'].map(school_name_lookup))
coverage['degree'] = coverage['degree'].fillna(coverage['ipeds_degree_level'])
coverage['cohort_band'] = coverage['cohort_band_obs'].combine_first(coverage['cohort_band_ipeds'])
coverage['cip2'] = coverage['cip2_obs'].combine_first(coverage['cip2_ipeds'])
coverage['cip4'] = coverage['cip4_obs'].combine_first(coverage['cip4_ipeds'])
coverage['cip_title'] = coverage['cip_title'].combine_first(coverage['cip4'].map(cip4_title_lookup)).fillna('')

for col in ['raw_education_rows', 'total_records', 'has_cip', 'triple_matches', 'manual_matches', 'school_year_matches', 'high_confidence', 'ipeds_completions']:
    coverage[col] = pd.to_numeric(coverage[col], errors='coerce').fillna(0)

coverage['revelio_completions'] = coverage['has_cip']
coverage['observed_ipeds_completions'] = coverage['revelio_completions']
coverage['cip_rate'] = np.where(
    coverage['total_records'] > 0,
    100 * coverage['has_cip'] / coverage['total_records'],
    np.nan,
).round(1)
coverage['ipeds_obs_rate'] = np.where(
    coverage['ipeds_completions'] > 0,
    100 * coverage['observed_ipeds_completions'] / coverage['ipeds_completions'],
    np.nan,
).round(1)
coverage['high_confidence_rate'] = np.where(
    coverage['total_records'] > 0,
    100 * coverage['high_confidence'] / coverage['total_records'],
    np.nan,
).round(1)
coverage['coverage_source'] = np.select(
    [
        coverage['_merge'].eq('both'),
        coverage['_merge'].eq('left_only'),
        coverage['_merge'].eq('right_only'),
    ],
    ['ipeds_and_revelio', 'ipeds_only', 'revelio_only'],
    default='unknown',
)

coverage = coverage[[
    'unitid', 'ipeds_name', 'degree', 'ipeds_degree_level', 'cohort_year', 'cohort_band',
    'cip2', 'cip4', 'cip_code', 'cip_title',
    'raw_education_rows', 'total_records', 'has_cip', 'revelio_completions',
    'observed_ipeds_completions', 'ipeds_completions', 'ipeds_obs_rate',
    'cip_rate', 'avg_cip_prob', 'triple_matches', 'manual_matches',
    'school_year_matches', 'high_confidence', 'high_confidence_rate', 'coverage_source'
]].sort_values(['unitid', 'degree', 'cohort_year', 'cip_code']).reset_index(drop=True)

print(f'school_coverage_fact: {len(coverage):,} rows')
print(f"  IPEDS-only rows: {(coverage['coverage_source'] == 'ipeds_only').sum():,}")
print(f"  Revelio-only rows: {(coverage['coverage_source'] == 'revelio_only').sum():,}")
print('\nCoverage sample - Berkeley bachelor majors:')
sample = coverage[(coverage['ipeds_name']=='University of California-Berkeley') & (coverage['degree']=='Bachelors')].sort_values(['cohort_year','revelio_completions'], ascending=[False, False]).head(10)
print(sample[['cohort_year','cip_code','cip_title','revelio_completions','ipeds_completions','ipeds_obs_rate','avg_cip_prob','coverage_source']].to_string(index=False))

write_fact_parquet(coverage, OUT_DIR / 'school_coverage_fact.parquet')
print('\nSaved school_coverage_fact.parquet.')


IPEDS table unavailable; falling back to local comp build: Query: SELECT 1 AS ok FROM USER_CAELAN.TMP_MONTHLY.IPEDS_COMPLETIONS LIMIT 1 failed to execute


Using local IPEDS completions files (_A) for IPEDS denominators.
school_coverage_fact: 36,001 rows

Coverage sample — Harvard bachelor majors:
 cohort_year cip_code                                       cip_title  total_records  ipeds_completions  ipeds_obs_rate  cip_rate  avg_cip_prob  high_confidence_rate
        2025  11.0701                                Computer Science           1513                NaN             NaN     100.0         0.846                  64.6
        2025     None                                                            714                NaN             NaN       0.0           NaN                   0.0
        2025  30.2501                      Cognitive Science, General            661                NaN             NaN     100.0         0.629                  24.7
        2025  45.0601                              Economics, General            616                NaN             NaN     100.0         0.921                  84.4
        2025  26.0406      

In [11]:

# =============================================================
# CELL 10: weighted school_quality_fact + school_missing_grad_fact + demographics
# =============================================================
# Quality and demographics now use the same calibrated outcome weights as the
# rest of the position-based facts. Missing-grad diagnostics remain raw education counts.

schema_probe = sfClient.load_df(f"SELECT * FROM {EDUCATION_CIP} LIMIT 0")
education_columns = {c.lower() for c in schema_probe.columns}
has_startdate = 'startdate' in education_columns
no_grad_condition = "CASE WHEN enddate IS NULL AND startdate IS NULL THEN 1 ELSE 0 END" if has_startdate else "CASE WHEN enddate IS NULL THEN 1 ELSE 0 END"
no_grad_definition = 'no startdate and no enddate' if has_startdate else 'no enddate (startdate column unavailable)'
print(f'No-grad-date definition: {no_grad_definition}')

quality_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip_code, cip_title,
        salary, seniority, analysis_weight,
        named_employer_flag, unknown_employer_flag, same_school_employer_flag, career_employer_flag
    FROM {SCRATCH}.SCHOOL_OUTCOMES_EMPLOYER_ENRICHED
"""
quality_by_major = weighted_aggregate_sql(
    quality_source + "\nWHERE cip_code IS NOT NULL",
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title'],
    count_alias='observed_positions',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "ROUND(SUM(named_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS named_employer_n",
        "ROUND(SUM(unknown_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS unknown_employer_n",
        "ROUND(SUM(same_school_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS same_school_employer_n",
        "ROUND(SUM(career_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS career_employer_n",
    ],
)
quality_all_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        'ALL' AS cip2, 'ALL' AS cip4, 'ALL' AS cip_code, 'All majors' AS cip_title,
        salary, seniority, analysis_weight,
        named_employer_flag, unknown_employer_flag, same_school_employer_flag, career_employer_flag
    FROM {SCRATCH}.SCHOOL_OUTCOMES_EMPLOYER_ENRICHED
"""
quality_all_major = weighted_aggregate_sql(
    quality_all_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title'],
    count_alias='observed_positions',
    salary_count_alias='salary_obs',
    extra_aggs=[
        "ROUND(SUM(named_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS named_employer_n",
        "ROUND(SUM(unknown_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS unknown_employer_n",
        "ROUND(SUM(same_school_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS same_school_employer_n",
        "ROUND(SUM(career_employer_flag * COALESCE(analysis_weight, 1.0)), 0) AS career_employer_n",
    ],
)
quality = sfClient.load_df(f"""
    SELECT * FROM ({quality_by_major})
    UNION ALL
    SELECT * FROM ({quality_all_major})
""")
quality.columns = [c.lower() for c in quality.columns]
quality['named_employer_pct'] = (100 * quality['named_employer_n'] / quality['observed_positions']).round(2)
quality['unknown_employer_pct'] = (100 * quality['unknown_employer_n'] / quality['observed_positions']).round(2)
quality['same_school_employer_pct'] = (100 * quality['same_school_employer_n'] / quality['observed_positions']).round(2)
quality['career_employer_pct'] = (100 * quality['career_employer_n'] / quality['observed_positions']).round(2)
quality['salary_obs_pct'] = (100 * quality['salary_obs'] / quality['observed_positions']).round(2)

print(f'school_quality_fact: {len(quality):,} rows')
write_fact_parquet(quality, OUT_DIR / 'school_quality_fact.parquet')
print('\nSaved school_quality_fact.parquet')

missing_grad = sfClient.load_df(f"""
    WITH base AS (
        SELECT
            e.unitid,
            e.ipeds_name,
            {degree_label_sql('e')} AS degree,
            LEFT({assigned_cip4_sql('e')}, 2) AS cip2,
            {assigned_cip4_sql('e')} AS cip4,
            {assigned_cip4_sql('e')} AS cip_code,
            {assigned_cip_title_sql('e')} AS cip_title,
            {no_grad_condition} AS no_grad_flag
        FROM {EDUCATION_CIP} e
        WHERE e.unitid IN ({UNITID_SQL})
          AND (e.degree IN ('Associate', 'Bachelor', 'Master', 'MBA') OR e.degree = 'Doctor')
    ),
    by_major AS (
        SELECT
            unitid,
            ipeds_name,
            degree,
            cip2,
            cip4,
            cip_code,
            MAX(cip_title) AS cip_title,
            COUNT(*) AS total_education_records,
            SUM(no_grad_flag) AS no_grad_date_n
        FROM base
        GROUP BY unitid, ipeds_name, degree, cip2, cip4, cip_code
    ),
    all_major AS (
        SELECT
            unitid,
            ipeds_name,
            degree,
            'ALL' AS cip2,
            'ALL' AS cip4,
            'ALL' AS cip_code,
            'All majors' AS cip_title,
            COUNT(*) AS total_education_records,
            SUM(no_grad_flag) AS no_grad_date_n
        FROM base
        GROUP BY unitid, ipeds_name, degree
    )
    SELECT * FROM by_major WHERE cip_code IS NOT NULL
    UNION ALL
    SELECT * FROM all_major
""")
missing_grad.columns = [c.lower() for c in missing_grad.columns]
missing_grad['no_grad_date_pct'] = (100 * missing_grad['no_grad_date_n'] / missing_grad['total_education_records']).round(2)
missing_grad['grad_date_available_n'] = missing_grad['total_education_records'] - missing_grad['no_grad_date_n']
write_fact_parquet(missing_grad, OUT_DIR / 'school_missing_grad_fact.parquet')
print('\nSaved school_missing_grad_fact.parquet')

DEMOGRAPHICS_TABLE = 'CLIENT_STANDARD.REVELIO_INTERNAL.STANDARD_202303_INDIVIDUAL_USER'

demographic_source = f"""
    WITH demo AS (
        SELECT
            USER_ID,
            COALESCE(NULLIF(SEX_PREDICTED, ''), 'Unknown') AS sex_predicted,
            COALESCE(NULLIF(ETHNICITY_PREDICTED, ''), 'Unknown') AS ethnicity_predicted,
            PRESTIGE
        FROM {DEMOGRAPHICS_TABLE}
    ),
    joined AS (
        SELECT
            b.*, d.sex_predicted, d.ethnicity_predicted, d.prestige
        FROM {SCRATCH}.SCHOOL_OUTCOMES_BASE b
        LEFT JOIN demo d
          ON b.user_id = d.user_id
    )
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip4 AS cip_code, cip_title,
        'sex' AS demographic_type,
        sex_predicted AS demographic_value,
        salary, analysis_weight, prestige
    FROM joined
    UNION ALL
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        cip2, cip4, cip4 AS cip_code, cip_title,
        'ethnicity' AS demographic_type,
        ethnicity_predicted AS demographic_value,
        salary, analysis_weight, prestige
    FROM joined
"""
demo_by_major = weighted_aggregate_sql(
    f"SELECT * FROM ({demographic_source}) d WHERE cip_code IS NOT NULL",
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'demographic_type', 'demographic_value'],
    count_alias='n_users',
    salary_count_alias='n_salary',
    include_seniority=False,
    extra_aggs=[
        "ROUND(SUM(CASE WHEN prestige IS NOT NULL THEN prestige * COALESCE(analysis_weight, 1.0) ELSE 0 END) / NULLIF(SUM(CASE WHEN prestige IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END), 0), 3) AS avg_prestige",
    ],
)
demo_all_source = f"""
    SELECT
        user_id, unitid, ipeds_name, degree, horizon, cohort_year, cohort_band,
        'ALL' AS cip2, 'ALL' AS cip4, 'ALL' AS cip_code, 'All majors' AS cip_title,
        demographic_type, demographic_value, salary, analysis_weight, prestige
    FROM ({demographic_source}) d
"""
demo_all_major = weighted_aggregate_sql(
    demo_all_source,
    ['unitid', 'ipeds_name', 'degree', 'horizon', 'cohort_year', 'cohort_band', 'cip2', 'cip4', 'cip_code', 'cip_title', 'demographic_type', 'demographic_value'],
    count_alias='n_users',
    salary_count_alias='n_salary',
    include_seniority=False,
    extra_aggs=[
        "ROUND(SUM(CASE WHEN prestige IS NOT NULL THEN prestige * COALESCE(analysis_weight, 1.0) ELSE 0 END) / NULLIF(SUM(CASE WHEN prestige IS NOT NULL THEN COALESCE(analysis_weight, 1.0) ELSE 0 END), 0), 3) AS avg_prestige",
    ],
)
demographics = sfClient.load_df(f"""
    WITH stacked AS (
        SELECT * FROM ({demo_by_major})
        UNION ALL
        SELECT * FROM ({demo_all_major})
    ),
    totals AS (
        SELECT
            unitid, degree, horizon, cohort_year, cip2, cip4, cip_code, demographic_type,
            SUM(weighted_n) AS total_users
        FROM stacked
        GROUP BY unitid, degree, horizon, cohort_year, cip2, cip4, cip_code, demographic_type
    )
    SELECT
        s.*,
        ROUND(s.weighted_n / NULLIF(t.total_users, 0) * 100, 2) AS share_pct
    FROM stacked s
    JOIN totals t
      ON s.unitid = t.unitid
     AND s.degree = t.degree
     AND s.horizon = t.horizon
     AND s.cohort_year = t.cohort_year
     AND s.cip2 = t.cip2
     AND s.cip4 = t.cip4
     AND s.cip_code = t.cip_code
     AND s.demographic_type = t.demographic_type
""")
demographics.columns = [c.lower() for c in demographics.columns]
print(f'school_demographics_fact: {len(demographics):,} rows')
write_fact_parquet(demographics, OUT_DIR / 'school_demographics_fact.parquet')
print('\nSaved school_demographics_fact.parquet')


No-grad-date definition: no startdate and no enddate


school_quality_fact: 80,072 rows

Sample quality rows at Berkeley (Bachelors, 5yr, 2018, all majors):
 observed_positions  named_employer_pct  unknown_employer_pct  same_school_employer_pct  career_employer_pct  salary_obs_pct
               5839               91.11                  8.89                      4.02                87.09           100.0

Saved school_quality_fact.parquet

school_missing_grad_fact: 4,055 rows
Missing-grad rule: no startdate and no enddate

Sample missing-grad rows at Berkeley (Bachelors, all majors):
 total_education_records  no_grad_date_n  no_grad_date_pct
                  300406           57771             19.23

Saved school_missing_grad_fact.parquet
school_demographics_fact: 410,378 rows

Sample demographics rows at Berkeley (Bachelors, 5yr, 2018, all majors):
demographic_type demographic_value  n_users share_pct  avg_prestige
             sex             empty      472      8.08        34.480
             sex              None     1092     18.70     

In [12]:
# =============================================================
# CELL 11: Export data products for the explorer / wireframe
# =============================================================
# Export one JSON per fact table + reference data for CIP titles.
# Parquet is the primary format; JSON powers the HTML prototype.

from decimal import Decimal

# Keep browser-facing JSON parts comfortably below common transfer/cache limits.
FACT_ROWS_PER_FILE = 5000

def _json_safe_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return float(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (pd.Timestamp,)):
        return v.isoformat()
    return v

def write_records_json(df, filename, chunk_size=5000):
    path = OUT_DIR / filename
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with open(tmp_path, 'w', encoding='utf-8') as f:
        f.write('[')
        first = True
        for start in range(0, len(df), chunk_size):
            chunk = df.iloc[start:start + chunk_size].copy()
            records = []
            for row in chunk.itertuples(index=False, name=None):
                record = {col: _json_safe_value(val) for col, val in zip(chunk.columns, row)}
                records.append(record)
            for record in records:
                if not first:
                    f.write(',')
                json.dump(record, f, ensure_ascii=False)
                first = False
        f.write(']')
    tmp_path.replace(path)
    size_mb = path.stat().st_size / 1024 / 1024
    print(f'  {filename}: {len(df):,} rows | {size_mb:.1f} MB')
    return path

def write_fact_json(df, filename, rows_per_file=FACT_ROWS_PER_FILE):
    stem = filename[:-5] if filename.endswith('.json') else filename
    single_path = OUT_DIR / filename
    for old in OUT_DIR.glob(f'{stem}.part*.json'):
        old.unlink()
    if single_path.exists():
        single_path.unlink()
    if len(df) <= rows_per_file:
        write_records_json(df.reset_index(drop=True), filename)
        return filename
    files = []
    for part_idx, start in enumerate(range(0, len(df), rows_per_file), start=1):
        part_name = f'{stem}.part{part_idx:03d}.json'
        write_records_json(df.iloc[start:start + rows_per_file].reset_index(drop=True), part_name)
        files.append(part_name)
    return files

# Load all fact tables from parquet
school_overview     = pd.read_parquet(OUT_DIR / 'school_overview_fact.parquet')
school_earnings      = pd.read_parquet(OUT_DIR / 'school_earnings_fact.parquet')
school_earnings_curve = pd.read_parquet(OUT_DIR / 'school_earnings_curve_fact.parquet')
school_employer      = pd.read_parquet(OUT_DIR / 'school_employer_fact.parquet')
school_employer_role = pd.read_parquet(OUT_DIR / 'school_employer_role_fact.parquet')
school_geo           = pd.read_parquet(OUT_DIR / 'school_geo_fact.parquet')
school_role          = pd.read_parquet(OUT_DIR / 'school_role_fact.parquet')
school_major         = pd.read_parquet(OUT_DIR / 'school_major_fact.parquet')
school_major_mix     = pd.read_parquet(OUT_DIR / 'school_major_mix_fact.parquet')
school_postgrad      = pd.read_parquet(OUT_DIR / 'school_postgrad_fact.parquet')
school_postgrad_flow = pd.read_parquet(OUT_DIR / 'school_postgrad_flow_fact.parquet')
school_postgrad_dest = pd.read_parquet(OUT_DIR / 'school_postgrad_destination_fact.parquet')
school_coverage      = pd.read_parquet(OUT_DIR / 'school_coverage_fact.parquet')
school_quality       = pd.read_parquet(OUT_DIR / 'school_quality_fact.parquet')
school_missing_grad  = pd.read_parquet(OUT_DIR / 'school_missing_grad_fact.parquet')
school_demographics  = pd.read_parquet(OUT_DIR / 'school_demographics_fact.parquet')

school_meta_records = school_meta[['unitid', 'ipeds_name']].to_dict('records')

# --- CIP reference tables ---
write_fact_parquet(cip2_df, OUT_DIR / 'cip2_titles.parquet')
write_fact_parquet(cip4_df, OUT_DIR / 'cip4_titles.parquet')
write_fact_parquet(cip6_df, OUT_DIR / 'cip6_titles.parquet')

print('\nWriting fact JSONs:')
fact_files = {
    'overview': write_fact_json(school_overview, 'school_overview_fact.json'),
    'earnings': write_fact_json(school_earnings, 'school_earnings_fact.json'),
    'earnings_curve': write_fact_json(school_earnings_curve, 'school_earnings_curve_fact.json'),
    'employers': write_fact_json(school_employer, 'school_employer_fact.json'),
    'employer_roles': write_fact_json(school_employer_role, 'school_employer_role_fact.json'),
    'geography': write_fact_json(school_geo, 'school_geo_fact.json'),
    'roles': write_fact_json(school_role, 'school_role_fact.json'),
    'majors': write_fact_json(school_major, 'school_major_fact.json'),
    'major_mix': write_fact_json(school_major_mix, 'school_major_mix_fact.json'),
    'postgrad': write_fact_json(school_postgrad, 'school_postgrad_fact.json'),
    'postgrad_flows': write_fact_json(school_postgrad_flow, 'school_postgrad_flow_fact.json'),
    'postgrad_destinations': write_fact_json(school_postgrad_dest, 'school_postgrad_destination_fact.json'),
    'coverage': write_fact_json(school_coverage, 'school_coverage_fact.json'),
    'quality': write_fact_json(school_quality, 'school_quality_fact.json'),
    'missing_grad': write_fact_json(school_missing_grad, 'school_missing_grad_fact.json'),
    'demographics': write_fact_json(school_demographics, 'school_demographics_fact.json'),
}

print('\nWriting reference JSONs:')
write_records_json(cip2_df, 'cip2_titles.json')
write_records_json(cip4_df, 'cip4_titles.json')
write_records_json(cip6_df, 'cip6_titles.json')

manifest = {
    'schools': school_meta_records,
    'facts': fact_files,
    'reference': {
        'cip2_titles': 'cip2_titles.json',
        'cip4_titles': 'cip4_titles.json',
        'cip6_titles': 'cip6_titles.json',
    },
    'grain_notes': {
        'earnings': 'school x degree x horizon x cohort_year x cip_level (ALL/CIP2/CIP4/CIP6)',
        'employers': 'school x degree x horizon x cohort_year x major x employer (full list for selected schools)',
        'employer_roles': 'school x degree x horizon x cohort_year x major x employer x role x industry',
        'geography': 'school x degree x horizon x cohort_year x major x location (full list for selected schools)',
        'roles': 'school x degree x horizon x cohort_year x major x role x industry',
        'majors': 'school x degree x horizon x cohort_year x cip4',
        'major_mix': 'school x degree x horizon x cohort_year x cip4 (composition only; includes projected current-student rows when available)',
        'postgrad': 'school x undergrad cohort_year x undergrad major',
        'postgrad_flows': 'school x undergrad cohort_year x undergrad major x first later degree level',
        'postgrad_destinations': 'school x undergrad cohort_year x undergrad major x first later degree destination',
        'coverage': 'IPEDS-scaffolded school x degree x cohort_year x cip4 with raw education rows, distinct Revelio completions, IPEDS denominators, and source flags',
        'quality': 'school x degree x horizon x cohort_year x major with observation funnel and employer-quality metrics',
        'missing_grad': 'school x degree x major with counts lacking any usable graduation date',
        'demographics': 'school x degree x horizon x cohort_year x major x demographic_type x demographic_value',
    }
}

manifest_path = OUT_DIR / 'explorer_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Wrote explorer_manifest.json')

explorer_data = {
    'schools': school_meta_records,
    'cip2_titles': dict(zip(cip2_df['cip2'], cip2_df['cip2_title'])),
    'cip4_titles': dict(zip(cip4_df['cip4'], cip4_df['cip4_title'])),
    'cip6_titles': dict(zip(cip6_df['cip6'], cip6_df['cip6_title'])),
}
with open(OUT_DIR / 'explorer_data.json', 'w') as f:
    json.dump(explorer_data, f)
print('\nWrote lightweight explorer_data.json (school list + CIP titles).')

total_output_mb = sum(p.stat().st_size for p in OUT_DIR.glob('*.json')) / 1024 / 1024
print(f'\nTotal output: {total_output_mb:.1f} MB across {len(list(OUT_DIR.glob("*.json")))} JSON files')



Writing fact JSONs:
  school_overview_fact.json: 3,002 rows | 1.1 MB
  school_earnings_fact.part001.json: 25,000 rows | 9.5 MB
  school_earnings_fact.part002.json: 25,000 rows | 9.5 MB
  school_earnings_fact.part003.json: 25,000 rows | 9.5 MB
  school_earnings_fact.part004.json: 12,158 rows | 4.6 MB
  school_earnings_curve_fact.part001.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part002.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part003.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part004.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part005.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part006.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part007.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part008.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part009.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part010.json: 25,000 rows | 10.6 MB
  school_earnings_curve_fact.part011.json: 25,00

  school_employer_fact.part001.json: 25,000 rows | 16.0 MB
  school_employer_fact.part002.json: 25,000 rows | 16.0 MB
  school_employer_fact.part003.json: 25,000 rows | 15.9 MB
  school_employer_fact.part004.json: 25,000 rows | 15.9 MB
  school_employer_fact.part005.json: 25,000 rows | 16.0 MB
  school_employer_fact.part006.json: 25,000 rows | 15.9 MB
  school_employer_fact.part007.json: 25,000 rows | 15.9 MB
  school_employer_fact.part008.json: 25,000 rows | 15.9 MB
  school_employer_fact.part009.json: 25,000 rows | 15.9 MB
  school_employer_fact.part010.json: 25,000 rows | 15.9 MB
  school_employer_fact.part011.json: 25,000 rows | 15.9 MB
  school_employer_fact.part012.json: 25,000 rows | 15.9 MB
  school_employer_fact.part013.json: 25,000 rows | 15.9 MB
  school_employer_fact.part014.json: 25,000 rows | 15.9 MB
  school_employer_fact.part015.json: 25,000 rows | 15.9 MB
  school_employer_fact.part016.json: 25,000 rows | 15.9 MB
  school_employer_fact.part017.json: 25,000 rows | 16.0 

In [13]:
# =============================================================
# CELL 12: Quick sanity checks across outputs
# =============================================================

checks = {
    'school_overview_fact': pd.read_parquet(OUT_DIR / 'school_overview_fact.parquet'),
    'school_earnings_fact': pd.read_parquet(OUT_DIR / 'school_earnings_fact.parquet'),
    'school_earnings_curve_fact': pd.read_parquet(OUT_DIR / 'school_earnings_curve_fact.parquet'),
    'school_employer_fact': pd.read_parquet(OUT_DIR / 'school_employer_fact.parquet'),
    'school_employer_role_fact': pd.read_parquet(OUT_DIR / 'school_employer_role_fact.parquet'),
    'school_geo_fact': pd.read_parquet(OUT_DIR / 'school_geo_fact.parquet'),
    'school_role_fact': pd.read_parquet(OUT_DIR / 'school_role_fact.parquet'),
    'school_major_fact': pd.read_parquet(OUT_DIR / 'school_major_fact.parquet'),
    'school_major_mix_fact': pd.read_parquet(OUT_DIR / 'school_major_mix_fact.parquet'),
    'school_postgrad_fact': pd.read_parquet(OUT_DIR / 'school_postgrad_fact.parquet'),
    'school_postgrad_flow_fact': pd.read_parquet(OUT_DIR / 'school_postgrad_flow_fact.parquet'),
    'school_postgrad_destination_fact': pd.read_parquet(OUT_DIR / 'school_postgrad_destination_fact.parquet'),
    'school_coverage_fact': pd.read_parquet(OUT_DIR / 'school_coverage_fact.parquet'),
    'school_quality_fact': pd.read_parquet(OUT_DIR / 'school_quality_fact.parquet'),
    'school_missing_grad_fact': pd.read_parquet(OUT_DIR / 'school_missing_grad_fact.parquet'),
    'school_demographics_fact': pd.read_parquet(OUT_DIR / 'school_demographics_fact.parquet'),
}

print('Row counts:')
for name, df in checks.items():
    print(f'  {name}: {len(df):,}')

print('\nDimension coverage checks:')
print('  overview has conventional doctorate labels:', any(str(d) in {'PhD', 'LAW', 'MD', 'Professional Doctorate', 'Other Doctorate', 'Doctorate'} for d in checks['school_overview_fact']['degree'].dropna().astype(str).unique()))
print('  overview cohort_year range:', checks['school_earnings_fact']['cohort_year'].min(), '-', checks['school_earnings_fact']['cohort_year'].max())
print('  annual earnings curve cohort_year range:', checks['school_earnings_curve_fact']['cohort_year'].min(), '-', checks['school_earnings_curve_fact']['cohort_year'].max())
print('  annual earnings curve horizon range:', checks['school_earnings_curve_fact']['horizon'].min(), '-', checks['school_earnings_curve_fact']['horizon'].max())
print('  annual earnings curve has class of 2025:', (checks['school_earnings_curve_fact']['cohort_year'] == 2025).any())
print('  major cohort_year range:', checks['school_major_fact']['cohort_year'].min(), '-', checks['school_major_fact']['cohort_year'].max())
print('  major_mix cohort_year range:', checks['school_major_mix_fact']['cohort_year'].min(), '-', checks['school_major_mix_fact']['cohort_year'].max())
print('  major_mix has projected current students:', 'current_student_flag' in checks['school_major_mix_fact'].columns and checks['school_major_mix_fact']['current_student_flag'].fillna(0).astype(int).eq(1).any())
if 'current_student_flag' in checks['school_major_mix_fact'].columns:
    current_mix = checks['school_major_mix_fact'][checks['school_major_mix_fact']['current_student_flag'].fillna(0).astype(int).eq(1)]
    if len(current_mix):
        print('  projected current-student class years:', sorted(current_mix['cohort_year'].dropna().astype(int).unique().tolist()))
print('  employer has ALL rows:', (checks['school_employer_fact']['cip_code'] == 'ALL').any())
print('  employer-role has employer grain:', 'employer' in checks['school_employer_role_fact'].columns)
print('  geo has ALL rows:', (checks['school_geo_fact']['cip_code'] == 'ALL').any())
print('  role has ALL rows:', (checks['school_role_fact']['cip_code'] == 'ALL').any())
print('  coverage has cohort_year:', 'cohort_year' in checks['school_coverage_fact'].columns)
print('  coverage has ipeds denominator:', 'ipeds_completions' in checks['school_coverage_fact'].columns)
print('  coverage has ipeds observation rate:', 'ipeds_obs_rate' in checks['school_coverage_fact'].columns)
print('  coverage has raw education rows:', 'raw_education_rows' in checks['school_coverage_fact'].columns)
print('  coverage has source flags:', 'coverage_source' in checks['school_coverage_fact'].columns)
if 'coverage_source' in checks['school_coverage_fact'].columns:
    print('  coverage source mix:', checks['school_coverage_fact']['coverage_source'].value_counts(dropna=False).to_dict())
print('  quality has career employer pct:', 'career_employer_pct' in checks['school_quality_fact'].columns)
print('  missing grad has no_grad_date_n:', 'no_grad_date_n' in checks['school_missing_grad_fact'].columns)
print('  demographics has share_pct:', 'share_pct' in checks['school_demographics_fact'].columns)
print('  postgrad has undergrad major:', 'undergrad_cip_code' in checks['school_postgrad_fact'].columns)
print('  postgrad flow has degree levels:', sorted(checks['school_postgrad_flow_fact']['postgrad_degree'].dropna().unique().tolist()))

print('\nSample Harvard major time series:')
harvard = checks['school_major_fact']
h_sample = harvard[(harvard['ipeds_name']=='University of California-Berkeley') &
                   (harvard['degree']=='Bachelors') &
                   (harvard['horizon']==5)]
print(h_sample[['cohort_year','cip_code','cip_title','n','median_salary']].sort_values(['cohort_year','n'], ascending=[True, False]).head(15).to_string(index=False))

coverage_check = checks['school_coverage_fact'].copy()
if {'ipeds_obs_rate', 'ipeds_completions', 'degree', 'cip_title', 'ipeds_name'}.issubset(coverage_check.columns):
    large_major_check = coverage_check[
        coverage_check['ipeds_completions'].fillna(0).astype(float).ge(25)
        & coverage_check['ipeds_obs_rate'].notna()
    ].copy()
    suspicious_high = large_major_check[large_major_check['ipeds_obs_rate'].astype(float).gt(140)].sort_values('ipeds_obs_rate', ascending=False).head(15)
    suspicious_low = large_major_check[large_major_check['ipeds_obs_rate'].astype(float).lt(30)].sort_values('ipeds_obs_rate').head(15)
    print('\nCoverage diagnostics:')
    print(f'  Large-major rows over 140% IPEDS coverage: {len(large_major_check[large_major_check["ipeds_obs_rate"].astype(float).gt(140)]):,}')
    if len(suspicious_high):
        print(suspicious_high[['ipeds_name','degree','cohort_year','cip_code','cip_title','revelio_completions','ipeds_completions','ipeds_obs_rate']].to_string(index=False))
    print(f'  Large-major rows under 30% IPEDS coverage: {len(large_major_check[large_major_check["ipeds_obs_rate"].astype(float).lt(30)]):,}')
    if len(suspicious_low):
        print(suspicious_low[['ipeds_name','degree','cohort_year','cip_code','cip_title','revelio_completions','ipeds_completions','ipeds_obs_rate']].to_string(index=False))


Row counts:
  school_overview_fact: 3,002
  school_earnings_fact: 87,158
  school_earnings_curve_fact: 297,087
  school_employer_fact: 2,286,900
  school_employer_role_fact: 2,953,132
  school_geo_fact: 919,093
  school_role_fact: 2,090,757
  school_major_fact: 77,070
  school_major_mix_fact: 77,070
  school_postgrad_fact: 10,311
  school_postgrad_flow_fact: 22,150
  school_postgrad_destination_fact: 150,477
  school_coverage_fact: 36,001
  school_quality_fact: 80,072
  school_missing_grad_fact: 4,055
  school_demographics_fact: 410,378

Dimension coverage checks:
  overview has conventional doctorate labels: True
  overview cohort_year range: 2005 - 2024
  annual earnings curve cohort_year range: 2005 - 2025
  annual earnings curve horizon range: 0 - 10
  annual earnings curve has class of 2025: True
  major cohort_year range: 2005 - 2025
  employer has ALL rows: True
  employer-role has employer grain: True
  geo has ALL rows: True
  role has ALL rows: True
  coverage has cohort_year

In [ ]:
# =============================================================
# CELL 13: API-ready platform Parquet export
# =============================================================
# Use the current platform export script from the app repo. This keeps the
# notebook from running a stale embedded export cell when the app schema moves.

PLATFORM_EXPORT_SCRIPT_PATHS = []
if os.environ.get('PLATFORM_EXPORT_SCRIPT'):
    PLATFORM_EXPORT_SCRIPT_PATHS.append(Path(os.environ['PLATFORM_EXPORT_SCRIPT']).expanduser())
PLATFORM_EXPORT_SCRIPT_PATHS.extend([
    DATA_DIR / 'platform_parquet_export.py',
    Path('platform_parquet_export.py'),
    Path('college_outcomes_platform/scripts/platform_parquet_export.py'),
    Path('scripts/platform_parquet_export.py'),
])

platform_export_script = next((path for path in PLATFORM_EXPORT_SCRIPT_PATHS if path.exists()), None)
if platform_export_script is None:
    raise FileNotFoundError('Could not find platform_parquet_export.py. Update PLATFORM_EXPORT_SCRIPT_PATHS to the app repo path.')

print(f'Running platform export script: {platform_export_script}')
exec(compile(platform_export_script.read_text(), str(platform_export_script), 'exec'))
